# UWHVF Spatio-Temporal Transformer for Glaucoma Progression Prediction (Kaggle Full-Spec)

This notebook implements a fully reproducible, Kaggle-ready research pipeline for glaucoma visual field (VF) progression prediction from the UWHVF `alldata.json` dataset. It automatically discovers the dataset path inside Kaggle, exports manuscript-grade artifacts to `/kaggle/working`, preserves strict patient-level leakage control, and supports deterministic CPU or GPU execution depending on the selected runtime. The core methodological contribution is a compact **hybrid dual-branch Spatio-Temporal Transformer (ST-Transformer)** that operates directly on raw `8x9` Humphrey Visual Field grids in right-eye orientation, encodes **TD** and **HVF sensitivity** through separate spatial branches, fuses them with **gated cross-modal fusion**, augments them with a **tabular biomarker adapter**, models longitudinal dynamics with an age-aware temporal Transformer, jointly predicts future MD slope and fast progression status, and quantifies uncertainty with **evidential regression plus MC-Dropout**.

Methodological novelty emphasized throughout the code:

- Raw-grid modeling instead of hand-crafted global indices only.
- Joint spatial-temporal representation learning on VF trajectories.
- Leakage-aware forward prediction using prefix visits to predict **future** MD slope.
- Multi-task learning for continuous progression rate and clinically meaningful fast-progressor classification.
- Uncertainty-aware predictions for translational deployment.
- Patient-level grouped splitting with severity/progression balancing and fairness auditing.

```mermaid
```mermaid
flowchart LR
    A["Prefix Visits per Eye<br/>TD 8x9, HVF 8x9, age"] --> B1["TD Spatial Encoder<br/>CNN + Self-Attention"]
    A --> B2["HVF Spatial Encoder<br/>CNN + Self-Attention"]
    A --> B3["Tabular Biomarker Adapter<br/>MD, PSD, VFI, trajectory features"]
    B1 --> C["Cross-Modal Gated Fusion"]
    B2 --> C
    C --> D["Age-Aware Temporal Transformer"]
    B3 --> E["Tabular-Gated Representation Fusion"]
    D --> E
    E --> E1["Evidential Regression Head<br/>Future MD Slope"]
    E --> E2["Binary Classification Head<br/>Fast vs Slow Progressor"]
    E1 --> F["Predictive Uncertainty<br/>Aleatoric + Epistemic"]
```

```mermaid
```mermaid
flowchart TD
    A["Load `alldata.json` exactly as stored"] --> B["Build patient-eye trajectories sorted by age"]
    B --> C["Create leakage-aware prefix-to-future samples"]
    C --> D["Patient-level grouped split<br/>80 / 10 / 10"]
    D --> E["Fit normalizers on training split only"]
    E --> F["Train ST-Transformer + ablations on Kaggle CPU/GPU"]
    F --> G["Evaluate slope, fast progression, C-index, calibration, decision curves"]
    G --> H["Explainability, fairness audit, LaTeX tables, publication figures"]
```

## Cell 0 - Setup



In [ ]:
# Kaggle quick start:
# 1. Upload this notebook to Kaggle.
# 2. Attach a dataset that contains `alldata.json`.
# 3. For the strongest full-spec run, enable a GPU accelerator.
# 4. Run all cells from top to bottom.
#
# If one or more packages are missing in Kaggle, uncomment once:
# %pip install -q --upgrade optuna shap lightgbm xgboost lifelines catboost

import copy
import json
import math
import os
import platform
import random
import sys
import textwrap
import warnings

from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

from IPython.display import Markdown, display
from scipy import stats
from scipy.stats import linregress
from sklearn.calibration import CalibrationDisplay
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    explained_variance_score,
    f1_score,
    log_loss,
    matthews_corrcoef,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    r2_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

try:
    import shap
except ImportError:
    shap = None

try:
    from lifelines import KaplanMeierFitter
    from lifelines.utils import concordance_index
    LIFELINES_AVAILABLE = True
except ImportError:
    LIFELINES_AVAILABLE = False

    def concordance_index(
        event_times: Sequence[float],
        predicted_scores: Sequence[float],
        event_observed: Optional[Sequence[float]] = None,
    ) -> float:
        event_times = np.asarray(event_times, dtype=float)
        predicted_scores = np.asarray(predicted_scores, dtype=float)
        if event_observed is None:
            event_observed = np.ones_like(event_times, dtype=int)
        else:
            event_observed = np.asarray(event_observed, dtype=int)

        concordant = 0.0
        tied = 0.0
        admissible = 0.0

        for i in range(len(event_times)):
            for j in range(i + 1, len(event_times)):
                ti, tj = event_times[i], event_times[j]
                pi, pj = predicted_scores[i], predicted_scores[j]
                ei, ej = event_observed[i], event_observed[j]

                if ti == tj:
                    continue

                if ti < tj and ei == 1:
                    admissible += 1.0
                    if pi < pj:
                        concordant += 1.0
                    elif pi == pj:
                        tied += 1.0
                elif tj < ti and ej == 1:
                    admissible += 1.0
                    if pj < pi:
                        concordant += 1.0
                    elif pi == pj:
                        tied += 1.0

        if admissible == 0:
            return float("nan")
        return float((concordant + 0.5 * tied) / admissible)

    class KaplanMeierFitter:
        """Minimal fallback Kaplan-Meier implementation for environments without lifelines."""

        def fit(
            self,
            durations: Sequence[float],
            event_observed: Optional[Sequence[float]] = None,
            label: Optional[str] = None,
        ) -> "KaplanMeierFitter":
            durations = np.asarray(durations, dtype=float)
            if event_observed is None:
                event_observed = np.ones_like(durations, dtype=int)
            else:
                event_observed = np.asarray(event_observed, dtype=int)

            unique_event_times = np.sort(np.unique(durations[event_observed == 1]))
            timeline = [0.0]
            survival = [1.0]
            current_survival = 1.0

            for event_time in unique_event_times:
                at_risk = np.sum(durations >= event_time)
                n_events = np.sum((durations == event_time) & (event_observed == 1))
                if at_risk > 0:
                    current_survival *= (1.0 - (n_events / at_risk))
                timeline.append(float(event_time))
                survival.append(float(current_survival))

            self.timeline_ = np.asarray(timeline, dtype=float)
            self.survival_function_ = np.asarray(survival, dtype=float)
            self.label = label or "KM estimate"
            return self

        def plot_survival_function(self, ax=None, ci_show: bool = False):
            if ax is None:
                _, ax = plt.subplots(1, 1, figsize=(8, 5))
            ax.step(self.timeline_, self.survival_function_, where="post", label=self.label)
            return ax

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

try:
    import xgboost as xgb
except ImportError:
    xgb = None

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
except ImportError:
    CatBoostClassifier = None
    CatBoostRegressor = None


warnings.filterwarnings("ignore")


SEED = 2026


def running_on_kaggle() -> bool:
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()


RUNNING_ON_KAGGLE = running_on_kaggle()
KAGGLE_DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/sanimyousuffahim/uwhvf-humphrey-visual-field-dataset"),
    Path("/kaggle/input/uwhvf-humphrey-visual-field-dataset"),
]
LOCAL_DATASET_ROOT_CANDIDATES = [
    Path(r"C:\Users\USER\Downloads"),
    Path.cwd(),
]


def resolve_data_path(filename: str = "alldata.json") -> Path:
    candidates = [
        Path(filename),
        Path.cwd() / filename,
    ]
    for local_root in LOCAL_DATASET_ROOT_CANDIDATES:
        candidates.append(local_root / filename)
    if RUNNING_ON_KAGGLE:
        for kaggle_root in KAGGLE_DATASET_ROOT_CANDIDATES:
            candidates.append(kaggle_root / filename)
        kaggle_root = Path("/kaggle/input")
        if kaggle_root.exists():
            candidates.extend(sorted(kaggle_root.rglob(filename)))
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched_roots = ["current working directory"]
    if RUNNING_ON_KAGGLE:
        searched_roots.append("/kaggle/input")
    raise FileNotFoundError(
        f"Could not find `{filename}`. Attach a Kaggle dataset containing `{filename}`. "
        f"Searched in: {', '.join(searched_roots)}, plus prioritized Kaggle roots "
        f"{[str(path) for path in KAGGLE_DATASET_ROOT_CANDIDATES]} and local roots "
        f"{[str(path) for path in LOCAL_DATASET_ROOT_CANDIDATES]}."
    )


def resolve_output_dir(default_name: str) -> Path:
    if RUNNING_ON_KAGGLE:
        return Path("/kaggle/working") / default_name
    return Path(default_name)


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


def choose_device(device_mode: str = "auto") -> torch.device:
    normalized = device_mode.lower()
    if normalized not in {"auto", "cpu", "cuda"}:
        raise ValueError(f"Unknown device_mode: {device_mode}")
    if normalized == "cpu":
        return torch.device("cpu")
    if normalized == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("`device_mode='cuda'` was requested but CUDA is unavailable.")
        return torch.device("cuda")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def package_version(package_name: str) -> str:
    try:
        from importlib.metadata import version

        return version(package_name)
    except Exception:
        return "not-installed"


@dataclass
class Config:
    data_path: Path = Path("alldata.json")
    output_dir: Path = Path("artifacts_uwhvf_st_transformer_kaggle")
    compute_profile: str = "kaggle_fullspec"
    device_mode: str = "auto"
    min_prefix_len: int = 2
    min_future_visits: int = 2
    min_total_visits: int = 4
    min_future_span_years: float = 0.5
    max_seq_len: int = 10
    batch_size: int = 16
    max_epochs: int = 40
    patience: int = 8
    optuna_trials: int = 50
    optuna_epochs: int = 18
    fast_progressor_threshold: float = -1.0
    event_drop_db: float = 3.0
    clip_td_min: float = -35.0
    clip_td_max: float = 10.0
    clip_hvf_min: float = 0.0
    clip_hvf_max: float = 40.0
    mc_dropout_passes: int = 8
    default_embed_dim: int = 128
    objective_train_fraction: float = 1.0
    objective_val_fraction: float = 1.0
    cache_dataset: bool = True
    train_all_prefixes: bool = True
    train_boosting_baseline: bool = True
    train_catboost_baseline: bool = True
    train_deep_baselines: bool = True
    optuna_timeout_seconds: Optional[int] = None
    bootstrap_replicates: int = 1000
    repeated_split_replicates: int = 5
    enable_repeated_tabular_validation: bool = True
    fast_sample_weight: float = 2.5
    extreme_sample_weight: float = 3.5
    severe_baseline_weight: float = 1.4
    extreme_slope_threshold: float = -1.5
    mild_slope_threshold: float = -0.5
    calibrate_probabilities: bool = True
    calibrate_regression: bool = True
    export_vector_figures: bool = True
    figure_dpi: int = 320


CONFIG = Config()

PROFILE_PRESETS = {
    "low_spec": {
        "device_mode": "cpu",
        "max_seq_len": 6,
        "batch_size": 8,
        "max_epochs": 14,
        "patience": 4,
        "optuna_trials": 6,
        "optuna_epochs": 4,
        "mc_dropout_passes": 3,
        "default_embed_dim": 64,
        "objective_train_fraction": 0.35,
        "objective_val_fraction": 0.50,
        "cache_dataset": True,
        "train_all_prefixes": False,
        "train_boosting_baseline": True,
        "train_catboost_baseline": False,
        "train_deep_baselines": False,
        "optuna_timeout_seconds": 1800,
        "bootstrap_replicates": 250,
        "repeated_split_replicates": 2,
        "enable_repeated_tabular_validation": False,
        "figure_dpi": 220,
    },
    "paper": {
        "device_mode": "cpu",
        "max_seq_len": 10,
        "batch_size": 16,
        "max_epochs": 40,
        "patience": 8,
        "optuna_trials": 50,
        "optuna_epochs": 18,
        "mc_dropout_passes": 8,
        "default_embed_dim": 128,
        "objective_train_fraction": 1.0,
        "objective_val_fraction": 1.0,
        "cache_dataset": True,
        "train_all_prefixes": True,
        "train_boosting_baseline": True,
        "train_catboost_baseline": True,
        "train_deep_baselines": True,
        "optuna_timeout_seconds": None,
        "bootstrap_replicates": 1000,
        "repeated_split_replicates": 5,
        "enable_repeated_tabular_validation": True,
        "figure_dpi": 320,
    },
    "kaggle_debug": {
        "device_mode": "auto",
        "max_seq_len": 8,
        "batch_size": 12,
        "max_epochs": 12,
        "patience": 4,
        "optuna_trials": 4,
        "optuna_epochs": 3,
        "mc_dropout_passes": 4,
        "default_embed_dim": 64,
        "objective_train_fraction": 0.40,
        "objective_val_fraction": 0.60,
        "cache_dataset": True,
        "train_all_prefixes": False,
        "train_boosting_baseline": True,
        "train_catboost_baseline": False,
        "train_deep_baselines": False,
        "optuna_timeout_seconds": 1200,
        "bootstrap_replicates": 200,
        "repeated_split_replicates": 2,
        "enable_repeated_tabular_validation": False,
        "figure_dpi": 220,
    },
    "kaggle_fullspec": {
        "device_mode": "auto",
        "max_seq_len": 10,
        "batch_size": 24,
        "max_epochs": 48,
        "patience": 10,
        "optuna_trials": 24,
        "optuna_epochs": 12,
        "mc_dropout_passes": 10,
        "default_embed_dim": 128,
        "objective_train_fraction": 1.0,
        "objective_val_fraction": 1.0,
        "cache_dataset": True,
        "train_all_prefixes": True,
        "train_boosting_baseline": True,
        "train_catboost_baseline": True,
        "train_deep_baselines": True,
        "optuna_timeout_seconds": 21600,
        "bootstrap_replicates": 1000,
        "repeated_split_replicates": 5,
        "enable_repeated_tabular_validation": True,
        "figure_dpi": 320,
    },
}

if CONFIG.compute_profile not in PROFILE_PRESETS:
    raise ValueError(f"Unknown compute_profile: {CONFIG.compute_profile}")

for key, value in PROFILE_PRESETS[CONFIG.compute_profile].items():
    setattr(CONFIG, key, value)

LOW_RESOURCE_PROFILES = {"low_spec", "kaggle_debug"}
FULLSPEC_PROFILES = {"paper", "kaggle_fullspec"}

CONFIG.data_path = resolve_data_path(CONFIG.data_path.name)
CONFIG.output_dir = resolve_output_dir(CONFIG.output_dir.name)
CONFIG.output_dir.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)

DEVICE = choose_device(CONFIG.device_mode)
MAX_CPU_THREADS = max(1, min(16 if RUNNING_ON_KAGGLE else 8, os.cpu_count() or 1))
torch.set_num_threads(MAX_CPU_THREADS)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "savefig.dpi": CONFIG.figure_dpi,
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.labelweight": "semibold",
        "legend.frameon": False,
    }
)

MODEL_COLORS = {
    "ST-Transformer": "#0B3C5D",
    "Hybrid Ensemble": "#0EAD69",
    "Constrained Ensemble": "#117A65",
    "LightGBM": "#F39C12",
    "XGBoost": "#D7263D",
    "ExtraTrees": "#5C4B51",
    "HistGB": "#3A86FF",
    "CatBoost": "#6A4C93",
    "LSTM": "#2A9D8F",
    "FT-Transformer": "#7A9E7E",
}

OPTIONAL_DEPENDENCY_STATUS = {
    "shap_available": shap is not None,
    "lifelines_available": LIFELINES_AVAILABLE,
    "lightgbm_available": lgb is not None,
    "xgboost_available": xgb is not None,
    "catboost_available": CatBoostRegressor is not None and CatBoostClassifier is not None,
}


def save_figure(fig: plt.Figure, stem: str, close: bool = False) -> None:
    stem_path = CONFIG.output_dir / stem
    fig.savefig(stem_path.with_suffix(".png"), bbox_inches="tight", facecolor="white", dpi=CONFIG.figure_dpi)
    if CONFIG.export_vector_figures:
        fig.savefig(stem_path.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    if close:
        plt.close(fig)


RUN_MANIFEST = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "running_on_kaggle": RUNNING_ON_KAGGLE,
    "platform": platform.platform(),
    "python_version": sys.version,
    "device": DEVICE.type,
    "cuda_available": bool(torch.cuda.is_available()),
    "max_cpu_threads": MAX_CPU_THREADS,
    "packages": {
        "torch": package_version("torch"),
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "scikit-learn": package_version("scikit-learn"),
        "scipy": package_version("scipy"),
        "matplotlib": package_version("matplotlib"),
        "seaborn": package_version("seaborn"),
        "optuna": package_version("optuna"),
        "shap": package_version("shap"),
        "lightgbm": package_version("lightgbm"),
        "xgboost": package_version("xgboost"),
        "lifelines": package_version("lifelines"),
        "catboost": package_version("catboost"),
    },
    "config": {
        key: str(value) if isinstance(value, Path) else value
        for key, value in asdict(CONFIG).items()
    },
}

with open(CONFIG.output_dir / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(RUN_MANIFEST, f, indent=2)

print("Running on Kaggle:", RUNNING_ON_KAGGLE)
print("Resolved data path:", CONFIG.data_path)
print("Output directory:", CONFIG.output_dir)
print("Device:", DEVICE)
print("Max CPU threads:", MAX_CPU_THREADS)
print("Optional dependency status:")
print(pd.Series(OPTIONAL_DEPENDENCY_STATUS, name="available").to_frame())
print(pd.Series(asdict(CONFIG), name="value").to_frame())
display(
    Markdown(
        textwrap.dedent(
            f"""
            **Compute profile:** `{CONFIG.compute_profile}`  
            **Device mode:** `{CONFIG.device_mode}` -> resolved to `{DEVICE.type}`  
            **Dataset path:** `{CONFIG.data_path}`  
            **Output directory:** `{CONFIG.output_dir}`

            `kaggle_fullspec` is the default publication-facing preset. If you only want a quick smoke test, switch to `kaggle_debug`.
            The resolver prioritizes the explicit Kaggle dataset root `/kaggle/input/datasets/sanimyousuffahim/uwhvf-humphrey-visual-field-dataset`.
            If `lifelines` is unavailable, the notebook uses an internal fallback for Kaplan-Meier plotting and concordance index so execution still succeeds.
            """
        )
    )
)



## Cell 1 - Data Loading and Exploration

This cell loads `alldata.json` exactly as distributed, inspects documented and hidden keys, derives classical VF indices from the raw TD matrix, and visualizes representative `8x9` fields. All values are stored in right-eye orientation, so we preserve that orientation everywhere for consistency.



In [ ]:
# Safety bootstrap: if the kernel was restarted and Cell 0 was not rerun,
# rebuild the minimal notebook state needed for data loading and downstream cells.
if "CONFIG" not in globals():
    import json
    import os
    import random
    import warnings

    from dataclasses import dataclass
    from pathlib import Path

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import seaborn as sns
    import torch

    warnings.filterwarnings("ignore")

    SEED = 2026
    KAGGLE_DATASET_ROOT_CANDIDATES = [
        Path("/kaggle/input/datasets/sanimyousuffahim/uwhvf-humphrey-visual-field-dataset"),
        Path("/kaggle/input/uwhvf-humphrey-visual-field-dataset"),
    ]
    LOCAL_DATASET_ROOT_CANDIDATES = [
        Path(r"C:\Users\USER\Downloads"),
        Path.cwd(),
    ]

    def running_on_kaggle() -> bool:
        return "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()

    def resolve_data_path(filename: str = "alldata.json") -> Path:
        candidates = [Path(filename), Path.cwd() / filename]
        candidates.extend([path / filename for path in LOCAL_DATASET_ROOT_CANDIDATES])
        if running_on_kaggle():
            candidates.extend([path / filename for path in KAGGLE_DATASET_ROOT_CANDIDATES])
            if Path("/kaggle/input").exists():
                candidates.extend(sorted(Path("/kaggle/input").rglob(filename)))
        for candidate in candidates:
            if candidate.exists():
                return candidate.resolve()
        raise FileNotFoundError(
            f"Could not find `{filename}` in the working directory, prioritized Kaggle roots "
            f"{[str(path) for path in KAGGLE_DATASET_ROOT_CANDIDATES]}, local roots "
            f"{[str(path) for path in LOCAL_DATASET_ROOT_CANDIDATES]}, or `/kaggle/input`."
        )

    def seed_everything(seed: int = SEED) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    @dataclass
    class Config:
        data_path: Path = Path("alldata.json")
        output_dir: Path = Path("artifacts_uwhvf_st_transformer_kaggle")
        compute_profile: str = "kaggle_fullspec"
        min_prefix_len: int = 2
        min_future_visits: int = 2
        min_total_visits: int = 4
        min_future_span_years: float = 0.5
        max_seq_len: int = 10
        train_all_prefixes: bool = True
        cache_dataset: bool = True
        fast_progressor_threshold: float = -1.0
        extreme_slope_threshold: float = -1.5
        fast_sample_weight: float = 2.5
        extreme_sample_weight: float = 3.5
        severe_baseline_weight: float = 1.4
        event_drop_db: float = 3.0
        clip_td_min: float = -35.0
        clip_td_max: float = 10.0
        clip_hvf_min: float = 0.0
        clip_hvf_max: float = 40.0

    seed_everything(SEED)
    CONFIG = Config()
    CONFIG.data_path = resolve_data_path(CONFIG.data_path.name)
    CONFIG.output_dir = (Path("/kaggle/working") if running_on_kaggle() else Path.cwd()) / CONFIG.output_dir.name
    CONFIG.output_dir.mkdir(parents=True, exist_ok=True)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sns.set_theme(style="whitegrid", context="talk")
    plt.rcParams.update({"figure.figsize": (10, 6), "savefig.dpi": 320})

    print("Rebuilt a minimal Kaggle-aware CONFIG because Cell 0 was not executed in the current kernel.")

with open(CONFIG.data_path, "r", encoding="utf-8") as f:
    dat = json.load(f)

required_top_keys = {"pts", "eyes", "hvfs", "data"}
missing_top_keys = required_top_keys.difference(dat.keys())
assert not missing_top_keys, f"Missing top-level keys: {missing_top_keys}"

coords = sorted(dat.get("coords", []), key=lambda item: item.get("index", 0))
valid_mask = np.zeros((8, 9), dtype=bool)
for item in coords:
    valid_mask[item["x"], item["y"]] = True

if valid_mask.sum() == 0:
    # Defensive fallback if `coords` is absent in another copy of the dataset.
    first_pid = next(iter(dat["data"]))
    first_eye_key = "R" if "R" in dat["data"][first_pid] else "L"
    first_td = np.asarray(dat["data"][first_pid][first_eye_key][0]["td"], dtype=float)
    valid_mask = first_td != 100

valid_coords = np.argwhere(valid_mask)
point_names = [f"vf_point_{i:02d}" for i in range(valid_mask.sum())]


def to_matrix(matrix_like: Sequence[Sequence[float]]) -> np.ndarray:
    """Convert a nested list into a float32 numpy array."""
    return np.asarray(matrix_like, dtype=np.float32)


def valid_values(matrix: np.ndarray) -> np.ndarray:
    """Return only valid VF values, excluding dataset filler value 100."""
    return matrix[matrix != 100]


def flatten_valid_points(matrix: np.ndarray) -> np.ndarray:
    """Flatten the 54 anatomically valid VF test points in dataset `coords` order."""
    flat = []
    for row, col in valid_coords:
        value = float(matrix[row, col])
        flat.append(np.nan if value == 100 else value)
    return np.asarray(flat, dtype=np.float32)


vfi_center = np.array([3.5, 4.0], dtype=np.float32)
vfi_distances = np.linalg.norm(valid_coords.astype(np.float32) - vfi_center[None, :], axis=1)
vfi_weights = 1.0 / (1.0 + vfi_distances)
vfi_weights = vfi_weights / vfi_weights.sum()


def compute_md(td_matrix: np.ndarray) -> float:
    """Mean deviation approximation from TD values over valid points."""
    vals = valid_values(td_matrix)
    return float(np.nanmean(vals)) if len(vals) else np.nan


def compute_psd(td_matrix: np.ndarray) -> float:
    """
    Pattern standard deviation approximation from TD.
    This is not the proprietary instrument PSD but a clinically interpretable surrogate.
    """
    vals = valid_values(td_matrix)
    if len(vals) < 2:
        return np.nan
    md = float(np.nanmean(vals))
    return float(np.sqrt(np.mean((vals - md) ** 2)))


def compute_vfi_approx(td_matrix: np.ndarray) -> float:
    """
    Weighted VFI-like preservation score.
    This is an explicit approximation, not the manufacturer's proprietary formula.
    """
    vals = flatten_valid_points(td_matrix)
    preservation = np.clip(1.0 + vals / 30.0, 0.0, 1.0)
    return float(100.0 * np.sum(vfi_weights * preservation))


def plot_vf_heatmap(ax, matrix: np.ndarray, title: str, cmap: str, vmin: float, vmax: float) -> None:
    """Visualize an 8x9 VF matrix while hiding structural filler cells."""
    masked = np.where(matrix == 100, np.nan, matrix)
    sns.heatmap(
        masked,
        mask=np.isnan(masked),
        ax=ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        square=True,
        cbar=True,
        linewidths=0.25,
        linecolor="black",
    )
    ax.set_title(title)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")


patient_key_counter = Counter()
record_key_counter = Counter()
gender_counter = Counter()
year_values = []
sequence_lengths = []

for patient_id, patient_entry in dat["data"].items():
    patient_key_counter.update(patient_entry.keys())
    gender_counter.update([patient_entry.get("gender", "").strip() or "UNK"])
    if "year" in patient_entry:
        year_values.append(patient_entry["year"])
    for eye in ["R", "L"]:
        if eye not in patient_entry:
            continue
        records = patient_entry[eye]
        sequence_lengths.append(len(records))
        for record in records:
            record_key_counter.update(record.keys())

print("Top-level keys:", list(dat.keys()))
print("Dataset counters:", {k: dat[k] for k in ["pts", "eyes", "hvfs"]})
print("Patient entry keys:", patient_key_counter)
print("Per-test record keys:", record_key_counter)
print("Gender distribution:", gender_counter)
print("Calendar year range:", (min(year_values), max(year_values)))
print("Sequence length summary:")
print(pd.Series(sequence_lengths).describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame("n_tests"))
print("Valid VF points from coords:", int(valid_mask.sum()))

sample_pid = next(iter(dat["data"]))
sample_patient = dat["data"][sample_pid]
sample_eye = "R" if "R" in sample_patient else "L"
sample_record = sample_patient[sample_eye][0]

print("\nExample patient id:", sample_pid)
print("Example patient-level keys:", list(sample_patient.keys()))
print("Example test-level keys:", list(sample_record.keys()))
print("Example age:", sample_record["age"])

sample_rows = []
for patient_id, patient_entry in dat["data"].items():
    for eye in ["R", "L"]:
        if eye not in patient_entry:
            continue
        for idx, record in enumerate(patient_entry[eye]):
            td = to_matrix(record["td"])
            hvf = to_matrix(record["hvf"])
            sample_rows.append(
                {
                    "patient_id": patient_id,
                    "eye": eye,
                    "visit_index": idx,
                    "age": float(record["age"]),
                    "MD": compute_md(td),
                    "PSD_approx": compute_psd(td),
                    "VFI_approx": compute_vfi_approx(td),
                    "mean_HVF": float(np.nanmean(valid_values(hvf))),
                }
            )
            if len(sample_rows) >= 5:
                break
        if len(sample_rows) >= 5:
            break
    if len(sample_rows) >= 5:
        break

display(pd.DataFrame(sample_rows))

td_example = to_matrix(sample_record["td"])
hvf_example = to_matrix(sample_record["hvf"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_vf_heatmap(axes[0], td_example, "Example TD Matrix", cmap="coolwarm", vmin=-35, vmax=5)
plot_vf_heatmap(axes[1], hvf_example, "Example HVF Sensitivity Matrix", cmap="viridis", vmin=0, vmax=35)
plt.tight_layout()
plt.show()

display(
    Markdown(
        textwrap.dedent(
            f"""
            **Exploration highlights**

            - The JSON contains the documented top-level keys plus an additional `coords` field describing the 54 valid VF points inside the `8x9` grid.
            - Patient entries include `gender` and `year` in addition to `R` / `L`, enabling fairness analysis and retrospective cohort characterization.
            - Test records include `hvf_seq` and `td_seq` in addition to the documented `age`, `hvf`, and `td`. The notebook does **not** rely on these hidden per-test vectors for prediction, keeping the main model grounded in raw `8x9` matrices.
            - Structural filler cells are always encoded as `100` and are masked throughout normalization, modeling, attention visualization, and metric computation.
            """
        )
    )
)



## Cell 2 - Preprocessing and Sequence Building

This cell constructs leakage-aware prefix-to-future prediction samples. For each patient-eye trajectory, visits are sorted by age, the first `k` visits become the model input, and the target is the **future** MD slope estimated from the remaining visits. The split is patient-level to prevent leakage across fellow eyes or multiple prefixes from the same person.



In [ ]:
def normalize_gender(raw_gender: Optional[str]) -> str:
    raw_gender = (raw_gender or "").strip().upper()
    if raw_gender in {"F", "M", "O"}:
        return raw_gender
    return "UNK"


def safe_linregress(x: Sequence[float], y: Sequence[float]) -> Optional[float]:
    """Stable wrapper around scipy linregress for short noisy clinical sequences."""
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if len(x) < 2 or np.ptp(x) <= 0:
        return None
    return float(linregress(x, y).slope)


central_mask = np.linalg.norm(valid_coords.astype(np.float32) - vfi_center[None, :], axis=1) <= 2.5
superior_mask = valid_coords[:, 0] < 4
inferior_mask = valid_coords[:, 0] >= 4


def visit_feature_vector(td_matrix: np.ndarray, hvf_matrix: np.ndarray) -> np.ndarray:
    """Derived per-visit clinical summary features used by the temporal and tabular baselines."""
    flat_td = flatten_valid_points(td_matrix)
    flat_hvf = flatten_valid_points(hvf_matrix)
    md = compute_md(td_matrix)
    psd = compute_psd(td_matrix)
    vfi = compute_vfi_approx(td_matrix)
    central_md = float(np.nanmean(flat_td[central_mask]))
    superior_md = float(np.nanmean(flat_td[superior_mask]))
    inferior_md = float(np.nanmean(flat_td[inferior_mask]))
    mean_hvf = float(np.nanmean(flat_hvf))
    return np.asarray(
        [md, psd, vfi, central_md, superior_md, inferior_md, mean_hvf],
        dtype=np.float32,
    )


def compute_time_to_event(
    future_visits: Sequence[Dict[str, np.ndarray]],
    reference_md: float,
    event_drop_db: float,
) -> Tuple[float, int]:
    """
    Event-based endpoint:
    time to first future visit with MD worse than the last observed prefix MD by >= event_drop_db.
    """
    ages = np.asarray([visit["age"] for visit in future_visits], dtype=np.float32)
    mds = np.asarray([visit["md"] for visit in future_visits], dtype=np.float32)
    if len(ages) == 0:
        return np.nan, 0
    event_indices = np.where(mds <= (reference_md - event_drop_db))[0]
    if len(event_indices) > 0:
        idx = int(event_indices[0])
        return float(ages[idx] - ages[0]), 1
    return float(ages[-1] - ages[0]), 0


def compute_clinical_sample_weight(future_slope: float, baseline_md: float) -> float:
    """
    Tail-aware sample weighting.
    Short VF sequences make extreme future slopes rare but clinically crucial; these
    weights help both deep and tree models spend capacity on fast progressors.
    """
    weight = 1.0
    if future_slope <= CONFIG.fast_progressor_threshold:
        weight *= CONFIG.fast_sample_weight
    if future_slope <= CONFIG.extreme_slope_threshold:
        weight *= CONFIG.extreme_sample_weight / CONFIG.fast_sample_weight
    if baseline_md <= -12.0:
        weight *= CONFIG.severe_baseline_weight
    return float(weight)


def build_tabular_features(
    prefix_visits: Sequence[Dict[str, np.ndarray]],
    gender: str,
) -> Tuple[np.ndarray, List[str]]:
    """
    Fixed-length summary feature vector for LightGBM/XGBoost and FT-Transformer baselines.
    Features include:
    - baseline / last / delta TD at each valid location
    - baseline / last / delta HVF at each valid location
    - classical indices and their prefix slopes
    - trajectory metadata and gender encoding
    """
    first_visit = prefix_visits[0]
    last_visit = prefix_visits[-1]

    first_td = first_visit["td_flat"]
    last_td = last_visit["td_flat"]
    delta_td = last_td - first_td

    first_hvf = first_visit["hvf_flat"]
    last_hvf = last_visit["hvf_flat"]
    delta_hvf = last_hvf - first_hvf

    prefix_ages = [visit["age"] for visit in prefix_visits]
    prefix_mds = [visit["md"] for visit in prefix_visits]
    prefix_psds = [visit["psd"] for visit in prefix_visits]
    prefix_vfis = [visit["vfi"] for visit in prefix_visits]

    md_slope = safe_linregress(prefix_ages, prefix_mds) or 0.0
    psd_slope = safe_linregress(prefix_ages, prefix_psds) or 0.0
    vfi_slope = safe_linregress(prefix_ages, prefix_vfis) or 0.0

    scalar_features = np.asarray(
        [
            len(prefix_visits),
            last_visit["age"] - first_visit["age"],
            first_visit["age"],
            first_visit["md"],
            last_visit["md"],
            last_visit["md"] - first_visit["md"],
            md_slope,
            first_visit["psd"],
            last_visit["psd"],
            last_visit["psd"] - first_visit["psd"],
            psd_slope,
            first_visit["vfi"],
            last_visit["vfi"],
            last_visit["vfi"] - first_visit["vfi"],
            vfi_slope,
            float(gender == "F"),
            float(gender == "M"),
            float(gender == "O"),
            float(gender == "UNK"),
        ],
        dtype=np.float32,
    )

    scalar_names = [
        "n_prefix_visits",
        "prefix_time_span",
        "baseline_age",
        "baseline_MD",
        "last_MD",
        "delta_MD",
        "prefix_MD_slope",
        "baseline_PSD",
        "last_PSD",
        "delta_PSD",
        "prefix_PSD_slope",
        "baseline_VFI",
        "last_VFI",
        "delta_VFI",
        "prefix_VFI_slope",
        "gender_F",
        "gender_M",
        "gender_O",
        "gender_UNK",
    ]

    td_names = (
        [f"td_first_{name}" for name in point_names]
        + [f"td_last_{name}" for name in point_names]
        + [f"td_delta_{name}" for name in point_names]
    )
    hvf_names = (
        [f"hvf_first_{name}" for name in point_names]
        + [f"hvf_last_{name}" for name in point_names]
        + [f"hvf_delta_{name}" for name in point_names]
    )

    feature_vector = np.concatenate([scalar_features, first_td, last_td, delta_td, first_hvf, last_hvf, delta_hvf])
    feature_names = scalar_names + td_names + hvf_names
    return feature_vector.astype(np.float32), feature_names


all_prefix_samples: List[Dict[str, object]] = []
excluded_eyes = defaultdict(int)

for patient_id, patient_entry in dat["data"].items():
    patient_gender = normalize_gender(patient_entry.get("gender"))
    patient_year = patient_entry.get("year", np.nan)

    for eye in ["R", "L"]:
        if eye not in patient_entry:
            continue

        raw_visits = sorted(patient_entry[eye], key=lambda record: float(record["age"]))
        cleaned_visits = []

        for visit_idx, record in enumerate(raw_visits):
            age = float(record["age"])
            if not np.isfinite(age) or age <= 0:
                # Extremely rare malformed ages are skipped to preserve temporal ordering.
                continue

            td_matrix = to_matrix(record["td"])
            hvf_matrix = to_matrix(record["hvf"])

            cleaned_visits.append(
                {
                    "age": age,
                    "td": td_matrix,
                    "hvf": hvf_matrix,
                    "td_flat": flatten_valid_points(td_matrix),
                    "hvf_flat": flatten_valid_points(hvf_matrix),
                    "visit_stats": visit_feature_vector(td_matrix, hvf_matrix),
                    "md": compute_md(td_matrix),
                    "psd": compute_psd(td_matrix),
                    "vfi": compute_vfi_approx(td_matrix),
                    "visit_index": visit_idx,
                }
            )

        if len(cleaned_visits) < CONFIG.min_total_visits:
            excluded_eyes["too_short"] += 1
            continue

        full_ages = [visit["age"] for visit in cleaned_visits]
        full_mds = [visit["md"] for visit in cleaned_visits]
        full_slope = safe_linregress(full_ages, full_mds)
        if full_slope is None:
            excluded_eyes["invalid_full_slope"] += 1
            continue

        eye_uid = f"{patient_id}_{eye}"

        max_prefix_len = min(CONFIG.max_seq_len, len(cleaned_visits) - CONFIG.min_future_visits)
        for prefix_len in range(CONFIG.min_prefix_len, max_prefix_len + 1):
            prefix_visits = cleaned_visits[:prefix_len]
            future_visits = cleaned_visits[prefix_len:]
            future_ages = [visit["age"] for visit in future_visits]
            future_mds = [visit["md"] for visit in future_visits]

            future_slope = safe_linregress(future_ages, future_mds)
            if future_slope is None:
                continue
            if (future_ages[-1] - future_ages[0]) < CONFIG.min_future_span_years:
                continue

            age_deltas = np.asarray([visit["age"] - prefix_visits[0]["age"] for visit in prefix_visits], dtype=np.float32)
            age_gaps = np.concatenate([[0.0], np.diff([visit["age"] for visit in prefix_visits])]).astype(np.float32)

            tabular_vector, tabular_feature_names = build_tabular_features(prefix_visits, patient_gender)
            event_time, event_observed = compute_time_to_event(
                future_visits=future_visits,
                reference_md=prefix_visits[-1]["md"],
                event_drop_db=CONFIG.event_drop_db,
            )
            sample_weight = compute_clinical_sample_weight(future_slope, prefix_visits[0]["md"])

            all_prefix_samples.append(
                {
                    "patient_id": patient_id,
                    "eye": eye,
                    "eye_uid": eye_uid,
                    "gender": patient_gender,
                    "year": patient_year,
                    "prefix_len": prefix_len,
                    "is_earliest_prefix": prefix_len == CONFIG.min_prefix_len,
                    "visits": prefix_visits,
                    "age_deltas": age_deltas,
                    "age_gaps": age_gaps,
                    "baseline_age": prefix_visits[0]["age"],
                    "baseline_md": prefix_visits[0]["md"],
                    "label_slope": future_slope,
                    "label_fast": int(future_slope <= CONFIG.fast_progressor_threshold),
                    "sample_weight": sample_weight,
                    "event_time": event_time,
                    "event_observed": event_observed,
                    "full_eye_slope": full_slope,
                    "tabular_vector": tabular_vector,
                    "tabular_feature_names": tabular_feature_names,
                }
            )

print("Excluded eyes by reason:", dict(excluded_eyes))
print("Total prefix samples:", len(all_prefix_samples))
print("Unique patients in sample pool:", len({sample["patient_id"] for sample in all_prefix_samples}))
print("Unique eyes in sample pool:", len({sample["eye_uid"] for sample in all_prefix_samples}))


def build_patient_strata(samples: Sequence[Dict[str, object]]) -> pd.DataFrame:
    """Aggregate patient-level statistics for balanced grouped splitting."""
    grouped = defaultdict(lambda: {"slopes": [], "baseline_mds": [], "seq_lens": [], "gender": "UNK"})
    for sample in samples:
        bucket = grouped[sample["patient_id"]]
        bucket["slopes"].append(float(sample["label_slope"]))
        bucket["baseline_mds"].append(float(sample["baseline_md"]))
        bucket["seq_lens"].append(int(sample["prefix_len"]))
        bucket["gender"] = sample["gender"]

    rows = []
    for patient_id, bucket in grouped.items():
        rows.append(
            {
                "patient_id": patient_id,
                "mean_future_slope": float(np.mean(bucket["slopes"])),
                "mean_baseline_md": float(np.mean(bucket["baseline_mds"])),
                "mean_prefix_len": float(np.mean(bucket["seq_lens"])),
                "gender": bucket["gender"],
            }
        )

    patient_df = pd.DataFrame(rows)
    patient_df["slope_bin"] = pd.qcut(
        patient_df["mean_future_slope"],
        q=min(4, patient_df["mean_future_slope"].nunique()),
        duplicates="drop",
        labels=False,
    )
    patient_df["severity_bin"] = pd.qcut(
        patient_df["mean_baseline_md"],
        q=min(4, patient_df["mean_baseline_md"].nunique()),
        duplicates="drop",
        labels=False,
    )
    patient_df["visit_bin"] = pd.qcut(
        patient_df["mean_prefix_len"],
        q=min(4, patient_df["mean_prefix_len"].nunique()),
        duplicates="drop",
        labels=False,
    )
    patient_df["strat_key"] = (
        patient_df["slope_bin"].astype(str)
        + "_"
        + patient_df["severity_bin"].astype(str)
        + "_"
        + patient_df["visit_bin"].astype(str)
    )

    rare_keys = patient_df["strat_key"].value_counts()
    rare_keys = rare_keys[rare_keys < 3].index
    patient_df.loc[patient_df["strat_key"].isin(rare_keys), "strat_key"] = "rare"
    return patient_df


def balanced_group_shuffle_split(
    patient_df: pd.DataFrame,
    test_size: float,
    random_state: int,
    n_candidates: int = 256,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Use GroupShuffleSplit repeatedly and keep the split whose strata distribution
    most closely matches the full cohort. This preserves patient-level grouping
    while approximating stratification by progression rate, baseline severity, and sequence length.
    """
    overall_dist = patient_df["strat_key"].value_counts(normalize=True)
    splitter = GroupShuffleSplit(
        n_splits=n_candidates,
        test_size=test_size,
        random_state=random_state,
    )
    best_score = np.inf
    best_pair = None

    for train_idx, test_idx in splitter.split(patient_df, groups=patient_df["patient_id"]):
        train_dist = (
            patient_df.iloc[train_idx]["strat_key"]
            .value_counts(normalize=True)
            .reindex(overall_dist.index, fill_value=0.0)
        )
        test_dist = (
            patient_df.iloc[test_idx]["strat_key"]
            .value_counts(normalize=True)
            .reindex(overall_dist.index, fill_value=0.0)
        )
        score = (train_dist - overall_dist).abs().sum() + (test_dist - overall_dist).abs().sum()
        if score < best_score:
            best_score = score
            best_pair = (train_idx, test_idx)

    return best_pair


patient_df = build_patient_strata(all_prefix_samples)
train_idx, temp_idx = balanced_group_shuffle_split(patient_df, test_size=0.20, random_state=SEED)
train_patients = set(patient_df.iloc[train_idx]["patient_id"])
temp_patient_df = patient_df.iloc[temp_idx].reset_index(drop=True)

val_idx, test_idx = balanced_group_shuffle_split(temp_patient_df, test_size=0.50, random_state=SEED + 1)
val_patients = set(temp_patient_df.iloc[val_idx]["patient_id"])
test_patients = set(temp_patient_df.iloc[test_idx]["patient_id"])

assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)


def select_split_samples(
    samples: Sequence[Dict[str, object]],
    patient_ids: set,
    split_name: str,
) -> List[Dict[str, object]]:
    """
    Training uses all eligible prefixes to maximize sample efficiency.
    Validation and test use only the earliest admissible prefix per eye to avoid
    overweighting long-followed eyes and to better mimic early-risk prediction.
    """
    subset = [sample for sample in samples if sample["patient_id"] in patient_ids]
    if split_name == "train" and CONFIG.train_all_prefixes:
        return subset

    earliest_per_eye = {}
    for sample in subset:
        current = earliest_per_eye.get(sample["eye_uid"])
        if current is None or sample["prefix_len"] < current["prefix_len"]:
            earliest_per_eye[sample["eye_uid"]] = sample
    return list(earliest_per_eye.values())


train_samples = select_split_samples(all_prefix_samples, train_patients, split_name="train")
val_samples = select_split_samples(all_prefix_samples, val_patients, split_name="val")
test_samples = select_split_samples(all_prefix_samples, test_patients, split_name="test")

print(
    {
        "train_samples": len(train_samples),
        "val_samples": len(val_samples),
        "test_samples": len(test_samples),
        "train_patients": len(train_patients),
        "val_patients": len(val_patients),
        "test_patients": len(test_patients),
    }
)


class FeatureNormalizer:
    """Fit normalization parameters on the training split only."""

    def __init__(self) -> None:
        self.td_mean = None
        self.td_std = None
        self.hvf_mean = None
        self.hvf_std = None
        self.visit_scaler = StandardScaler()
        self.tabular_scaler = StandardScaler()

    def fit(self, samples: Sequence[Dict[str, object]]) -> None:
        td_sum = np.zeros((8, 9), dtype=np.float64)
        td_sum_sq = np.zeros((8, 9), dtype=np.float64)
        td_count = np.zeros((8, 9), dtype=np.float64)
        hvf_sum = np.zeros((8, 9), dtype=np.float64)
        hvf_sum_sq = np.zeros((8, 9), dtype=np.float64)
        hvf_count = np.zeros((8, 9), dtype=np.float64)
        visit_stats = []
        tabular = []

        for sample in samples:
            tabular.append(sample["tabular_vector"])
            for visit in sample["visits"]:
                td = np.clip(visit["td"], CONFIG.clip_td_min, CONFIG.clip_td_max)
                hvf = np.clip(visit["hvf"], CONFIG.clip_hvf_min, CONFIG.clip_hvf_max)
                td_valid = visit["td"] != 100
                hvf_valid = visit["hvf"] != 100

                td_sum[td_valid] += td[td_valid]
                td_sum_sq[td_valid] += td[td_valid] ** 2
                td_count[td_valid] += 1

                hvf_sum[hvf_valid] += hvf[hvf_valid]
                hvf_sum_sq[hvf_valid] += hvf[hvf_valid] ** 2
                hvf_count[hvf_valid] += 1

                visit_stats.append(visit["visit_stats"])

        self.td_mean = td_sum / np.maximum(td_count, 1.0)
        self.hvf_mean = hvf_sum / np.maximum(hvf_count, 1.0)

        td_var = td_sum_sq / np.maximum(td_count, 1.0) - self.td_mean ** 2
        hvf_var = hvf_sum_sq / np.maximum(hvf_count, 1.0) - self.hvf_mean ** 2

        self.td_std = np.sqrt(np.clip(td_var, 1e-6, None))
        self.hvf_std = np.sqrt(np.clip(hvf_var, 1e-6, None))

        self.td_mean[~valid_mask] = 0.0
        self.td_std[~valid_mask] = 1.0
        self.hvf_mean[~valid_mask] = 0.0
        self.hvf_std[~valid_mask] = 1.0

        self.visit_scaler.fit(np.vstack(visit_stats))
        self.tabular_scaler.fit(np.vstack(tabular))

    def normalize_grid(self, grid: np.ndarray, modality: str) -> np.ndarray:
        grid = np.asarray(grid, dtype=np.float32).copy()
        valid = grid != 100
        if modality == "td":
            grid[valid] = np.clip(grid[valid], CONFIG.clip_td_min, CONFIG.clip_td_max)
            grid[valid] = (grid[valid] - self.td_mean[valid]) / self.td_std[valid]
        elif modality == "hvf":
            grid[valid] = np.clip(grid[valid], CONFIG.clip_hvf_min, CONFIG.clip_hvf_max)
            grid[valid] = (grid[valid] - self.hvf_mean[valid]) / self.hvf_std[valid]
        else:
            raise ValueError(f"Unsupported modality: {modality}")
        grid[~valid] = 0.0
        return grid.astype(np.float32)

    def normalize_visit_stats(self, values: np.ndarray) -> np.ndarray:
        return self.visit_scaler.transform(values.reshape(1, -1)).reshape(-1).astype(np.float32)

    def normalize_tabular(self, values: np.ndarray) -> np.ndarray:
        return self.tabular_scaler.transform(values.reshape(1, -1)).reshape(-1).astype(np.float32)


normalizer = FeatureNormalizer()
normalizer.fit(train_samples)


class VFSequenceDataset(torch.utils.data.Dataset):
    """PyTorch dataset for leakage-aware prefix-to-future VF progression prediction."""

    def __init__(
        self,
        samples: Sequence[Dict[str, object]],
        normalizer: FeatureNormalizer,
        cache_items: bool = True,
    ):
        self.samples = list(samples)
        self.normalizer = normalizer
        self.cache_items = cache_items
        self.cached_items = [self._build_item(sample) for sample in self.samples] if cache_items else None

    def __len__(self) -> int:
        return len(self.samples)

    def _build_item(self, sample: Dict[str, object]) -> Dict[str, object]:
        td_seq = []
        hvf_seq = []
        visit_stats_seq = []
        visit_vector_seq = []

        for visit_idx, visit in enumerate(sample["visits"]):
            td_norm = self.normalizer.normalize_grid(visit["td"], modality="td")
            hvf_norm = self.normalizer.normalize_grid(visit["hvf"], modality="hvf")
            visit_stats_norm = self.normalizer.normalize_visit_stats(visit["visit_stats"])

            flat_td_norm = td_norm[valid_mask]
            flat_hvf_norm = hvf_norm[valid_mask]
            visit_vector = np.concatenate(
                [
                    flat_td_norm,
                    flat_hvf_norm,
                    visit_stats_norm,
                    np.asarray([sample["age_deltas"][visit_idx], sample["age_gaps"][visit_idx]], dtype=np.float32),
                ]
            )

            td_seq.append(td_norm)
            hvf_seq.append(hvf_norm)
            visit_stats_seq.append(visit_stats_norm)
            visit_vector_seq.append(visit_vector)

        return {
            "patient_id": sample["patient_id"],
            "eye_uid": sample["eye_uid"],
            "gender": sample["gender"],
            "baseline_age": np.float32(sample["baseline_age"]),
            "baseline_md": np.float32(sample["baseline_md"]),
            "td_seq": np.stack(td_seq).astype(np.float32),
            "hvf_seq": np.stack(hvf_seq).astype(np.float32),
            "visit_stats_seq": np.stack(visit_stats_seq).astype(np.float32),
            "visit_vector_seq": np.stack(visit_vector_seq).astype(np.float32),
            "age_deltas": sample["age_deltas"].astype(np.float32),
            "age_gaps": sample["age_gaps"].astype(np.float32),
            "tabular_vector": self.normalizer.normalize_tabular(sample["tabular_vector"]),
            "label_slope": np.float32(sample["label_slope"]),
            "label_fast": np.float32(sample["label_fast"]),
            "sample_weight": np.float32(sample.get("sample_weight", 1.0)),
            "event_time": np.float32(sample["event_time"]),
            "event_observed": np.float32(sample["event_observed"]),
            "seq_len": len(sample["visits"]),
        }

    def __getitem__(self, index: int) -> Dict[str, object]:
        if self.cached_items is not None:
            return self.cached_items[index]
        return self._build_item(self.samples[index])


def collate_vf_sequences(batch: Sequence[Dict[str, object]]) -> Dict[str, object]:
    max_len = max(item["seq_len"] for item in batch)
    batch_size = len(batch)
    visit_stats_dim = batch[0]["visit_stats_seq"].shape[-1]
    visit_vector_dim = batch[0]["visit_vector_seq"].shape[-1]
    tabular_dim = batch[0]["tabular_vector"].shape[-1]

    td_tensor = torch.zeros(batch_size, max_len, 8, 9, dtype=torch.float32)
    hvf_tensor = torch.zeros(batch_size, max_len, 8, 9, dtype=torch.float32)
    visit_stats_tensor = torch.zeros(batch_size, max_len, visit_stats_dim, dtype=torch.float32)
    visit_vector_tensor = torch.zeros(batch_size, max_len, visit_vector_dim, dtype=torch.float32)
    age_deltas_tensor = torch.zeros(batch_size, max_len, dtype=torch.float32)
    age_gaps_tensor = torch.zeros(batch_size, max_len, dtype=torch.float32)
    padding_mask = torch.ones(batch_size, max_len, dtype=torch.bool)
    tabular_tensor = torch.zeros(batch_size, tabular_dim, dtype=torch.float32)
    baseline_age_tensor = torch.zeros(batch_size, dtype=torch.float32)
    baseline_md_tensor = torch.zeros(batch_size, dtype=torch.float32)
    label_slope_tensor = torch.zeros(batch_size, dtype=torch.float32)
    label_fast_tensor = torch.zeros(batch_size, dtype=torch.float32)
    sample_weight_tensor = torch.ones(batch_size, dtype=torch.float32)
    event_time_tensor = torch.zeros(batch_size, dtype=torch.float32)
    event_observed_tensor = torch.zeros(batch_size, dtype=torch.float32)
    seq_len_tensor = torch.zeros(batch_size, dtype=torch.long)
    valid_mask_tensor = torch.from_numpy(valid_mask.astype(np.float32)).repeat(batch_size, 1, 1)

    patient_ids, eye_uids, genders = [], [], []

    for idx, item in enumerate(batch):
        seq_len = item["seq_len"]
        td_tensor[idx, :seq_len] = torch.from_numpy(item["td_seq"])
        hvf_tensor[idx, :seq_len] = torch.from_numpy(item["hvf_seq"])
        visit_stats_tensor[idx, :seq_len] = torch.from_numpy(item["visit_stats_seq"])
        visit_vector_tensor[idx, :seq_len] = torch.from_numpy(item["visit_vector_seq"])
        age_deltas_tensor[idx, :seq_len] = torch.from_numpy(item["age_deltas"])
        age_gaps_tensor[idx, :seq_len] = torch.from_numpy(item["age_gaps"])
        padding_mask[idx, :seq_len] = False
        tabular_tensor[idx] = torch.from_numpy(item["tabular_vector"])
        baseline_age_tensor[idx] = torch.tensor(item["baseline_age"])
        baseline_md_tensor[idx] = torch.tensor(item["baseline_md"])
        label_slope_tensor[idx] = torch.tensor(item["label_slope"])
        label_fast_tensor[idx] = torch.tensor(item["label_fast"])
        sample_weight_tensor[idx] = torch.tensor(item["sample_weight"])
        event_time_tensor[idx] = torch.tensor(item["event_time"])
        event_observed_tensor[idx] = torch.tensor(item["event_observed"])
        seq_len_tensor[idx] = seq_len
        patient_ids.append(item["patient_id"])
        eye_uids.append(item["eye_uid"])
        genders.append(item["gender"])

    return {
        "patient_id": patient_ids,
        "eye_uid": eye_uids,
        "gender": genders,
        "td_seq": td_tensor,
        "hvf_seq": hvf_tensor,
        "visit_stats_seq": visit_stats_tensor,
        "visit_vector_seq": visit_vector_tensor,
        "age_deltas": age_deltas_tensor,
        "age_gaps": age_gaps_tensor,
        "padding_mask": padding_mask,
        "tabular_vector": tabular_tensor,
        "baseline_age": baseline_age_tensor,
        "baseline_md": baseline_md_tensor,
        "label_slope": label_slope_tensor,
        "label_fast": label_fast_tensor,
        "sample_weight": sample_weight_tensor,
        "event_time": event_time_tensor,
        "event_observed": event_observed_tensor,
        "seq_len": seq_len_tensor,
        "valid_mask": valid_mask_tensor,
    }


train_dataset = VFSequenceDataset(train_samples, normalizer, cache_items=CONFIG.cache_dataset)
val_dataset = VFSequenceDataset(val_samples, normalizer, cache_items=CONFIG.cache_dataset)
test_dataset = VFSequenceDataset(test_samples, normalizer, cache_items=CONFIG.cache_dataset)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_vf_sequences,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_vf_sequences,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_vf_sequences,
)

example_batch = next(iter(train_loader))
for key, value in example_batch.items():
    if torch.is_tensor(value):
        print(key, tuple(value.shape))

display(
    Markdown(
        textwrap.dedent(
            f"""
            **Sequence building summary**

            - Prediction task: from the first `k` visits, estimate the **future** MD slope from the remaining visits.
            - Minimum input visits: `{CONFIG.min_prefix_len}`
            - Minimum future visits: `{CONFIG.min_future_visits}`
            - Minimum total visits per usable eye: `{CONFIG.min_total_visits}`
            - Fast progressor threshold: slope `<= {CONFIG.fast_progressor_threshold:.1f}` dB/year
            - Event endpoint: first future MD drop of `>= {CONFIG.event_drop_db:.1f}` dB relative to the last observed prefix MD
            - Split strategy: patient-level grouped split with balancing over progression rate, baseline MD severity, and sequence length
            """
        )
    )
)



## Cell 3 - Model Definition

The ST-Transformer below is intentionally compact enough for deterministic CPU research while still scaling cleanly to Kaggle GPU acceleration for the full-spec runs. It uses **two spatial branches**: one for TD and one for HVF sensitivity. Each branch applies a lightweight CNN plus spatial self-attention to raw `8x9` grids, after which a **cross-modal gated fusion** module blends modality-specific information before the **age-aware temporal Transformer** models disease evolution. We then add a **tabular biomarker adapter** so the final representation benefits from strong global indices and trajectory summaries in addition to raw-grid structure. The shared hybrid representation feeds an **evidential regression head** for future slope and a binary classification head for fast progression.



In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim: int, mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()
        hidden_dim = int(dim * mlp_ratio)
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    """Minimal transformer block that optionally returns attention weights for paper figures."""

    def __init__(self, dim: int, num_heads: int, dropout: float = 0.1, mlp_ratio: float = 4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.drop_path = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = FeedForward(dim, mlp_ratio=mlp_ratio, dropout=dropout)

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
        need_weights: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        norm_x = self.norm1(x)
        attn_out, attn_weights = self.attn(
            norm_x,
            norm_x,
            norm_x,
            key_padding_mask=key_padding_mask,
            need_weights=need_weights,
            average_attn_weights=False,
        )
        x = x + self.drop_path(attn_out)
        x = x + self.ffn(self.norm2(x))
        return x, attn_weights


class AgeEncoding(nn.Module):
    """
    Continuous positional encoding based on age deltas and visit-to-visit gaps.
    This is preferable to integer index encoding because the UWHVF dataset lacks exact dates.
    """

    def __init__(self, dim: int, num_frequencies: int = 16):
        super().__init__()
        self.num_frequencies = num_frequencies
        self.proj = nn.Linear(2 + 4 * num_frequencies, dim)

    def forward(self, age_deltas: torch.Tensor, age_gaps: torch.Tensor) -> torch.Tensor:
        device = age_deltas.device
        frequencies = torch.exp(
            torch.linspace(math.log(1e-2), math.log(10.0), self.num_frequencies, device=device)
        )
        delta = age_deltas.unsqueeze(-1)
        gap = age_gaps.unsqueeze(-1)
        fourier_features = torch.cat(
            [
                delta,
                gap,
                torch.sin(delta * frequencies),
                torch.cos(delta * frequencies),
                torch.sin(gap * frequencies),
                torch.cos(gap * frequencies),
            ],
            dim=-1,
        )
        return self.proj(fourier_features)


class SingleModalitySpatialEncoder(nn.Module):
    """
    Spatial encoder for a single VF modality.
    Each branch sees one modality map plus the structural-validity mask,
    which empirically stabilizes CPU training on sparse `8x9` grids.
    """

    def __init__(
        self,
        embed_dim: int = 128,
        num_heads: int = 4,
        num_layers: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.cnn_stem = nn.Sequential(
            nn.Conv2d(2, embed_dim // 2, kernel_size=3, padding=1),
            nn.GELU(),
            nn.GroupNorm(1, embed_dim // 2),
            nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=1),
            nn.GELU(),
            nn.GroupNorm(1, embed_dim),
        )
        self.position_embedding = nn.Parameter(torch.zeros(1, 72, embed_dim))
        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, num_heads=num_heads, dropout=dropout) for _ in range(num_layers)]
        )
        self.pool_query = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pool_norm = nn.LayerNorm(embed_dim)
        self.pool_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        nn.init.normal_(self.position_embedding, mean=0.0, std=0.02)
        nn.init.normal_(self.pool_query, mean=0.0, std=0.02)

    def forward(
        self,
        modality_seq: torch.Tensor,
        valid_mask_tensor: torch.Tensor,
        return_attention: bool = False,
        prefix_name: str = "modality",
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        batch_size, seq_len, _, _ = modality_seq.shape
        valid_channel = valid_mask_tensor.unsqueeze(1).repeat(1, seq_len, 1, 1)
        x = torch.stack([modality_seq, valid_channel], dim=2)
        x = x.reshape(batch_size * seq_len, 2, 8, 9)
        x = self.cnn_stem(x).flatten(2).transpose(1, 2)
        x = x + self.position_embedding

        token_padding_mask = ~valid_mask_tensor.reshape(batch_size, -1).bool()
        token_padding_mask = token_padding_mask.unsqueeze(1).repeat(1, seq_len, 1).reshape(batch_size * seq_len, -1)

        spatial_attentions = []
        for block in self.blocks:
            x, attn_weights = block(
                x,
                key_padding_mask=token_padding_mask,
                need_weights=return_attention,
            )
            if return_attention:
                spatial_attentions.append(attn_weights)

        pooled_query = self.pool_query.expand(batch_size * seq_len, 1, self.embed_dim)
        pooled_tokens, pool_attn = self.pool_attn(
            self.pool_norm(pooled_query),
            self.pool_norm(x),
            self.pool_norm(x),
            key_padding_mask=token_padding_mask,
            need_weights=return_attention,
            average_attn_weights=False,
        )
        pooled_tokens = pooled_tokens.squeeze(1).reshape(batch_size, seq_len, self.embed_dim)

        extra = {}
        if return_attention:
            extra[f"{prefix_name}_spatial_attentions"] = spatial_attentions
            extra[f"{prefix_name}_spatial_pool_attn"] = pool_attn.reshape(
                batch_size,
                seq_len,
                pool_attn.shape[1],
                pool_attn.shape[-1],
            )
        return pooled_tokens, extra


class CrossModalGatedFusion(nn.Module):
    """
    Cross-modal gated fusion between TD and HVF visit embeddings.
    This gives the model a learnable mechanism to rely more heavily on TD
    when progression signal is deficit-driven, while still using HVF sensitivity
    when residual sensitivity structure is informative.
    """

    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Linear(dim * 2, dim)
        self.gate = nn.Sequential(
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.Sigmoid(),
        )
        self.norm = nn.LayerNorm(dim)

    def forward(
        self,
        td_tokens: torch.Tensor,
        hvf_tokens: torch.Tensor,
        return_attention: bool = False,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        fusion_features = torch.cat(
            [
                td_tokens,
                hvf_tokens,
                td_tokens - hvf_tokens,
                td_tokens * hvf_tokens,
            ],
            dim=-1,
        )
        gate = self.gate(fusion_features)
        fused = gate * td_tokens + (1.0 - gate) * hvf_tokens
        fused = self.norm(fused + self.proj(torch.cat([td_tokens, hvf_tokens], dim=-1)))
        extra = {}
        if return_attention:
            extra["fusion_gate"] = gate
        return fused, extra


class TemporalEncoder(nn.Module):
    """Temporal transformer over visit embeddings with continuous age encoding."""

    def __init__(
        self,
        spatial_dim: int,
        visit_stats_dim: int,
        model_dim: int = 128,
        num_heads: int = 4,
        num_layers: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.model_dim = model_dim
        self.input_proj = nn.Linear(spatial_dim + visit_stats_dim, model_dim)
        self.age_encoding = AgeEncoding(model_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, model_dim))
        self.blocks = nn.ModuleList(
            [TransformerBlock(model_dim, num_heads=num_heads, dropout=dropout) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(model_dim)
        nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

    def forward(
        self,
        spatial_tokens: torch.Tensor,
        visit_stats_seq: torch.Tensor,
        age_deltas: torch.Tensor,
        age_gaps: torch.Tensor,
        padding_mask: torch.Tensor,
        return_attention: bool = False,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        x = torch.cat([spatial_tokens, visit_stats_seq], dim=-1)
        x = self.input_proj(x) + self.age_encoding(age_deltas, age_gaps)

        cls_token = self.cls_token.expand(x.shape[0], 1, self.model_dim)
        x = torch.cat([cls_token, x], dim=1)
        cls_padding = torch.zeros((padding_mask.shape[0], 1), dtype=torch.bool, device=padding_mask.device)
        temporal_padding_mask = torch.cat([cls_padding, padding_mask], dim=1)

        temporal_attentions = []
        for block in self.blocks:
            x, attn_weights = block(
                x,
                key_padding_mask=temporal_padding_mask,
                need_weights=return_attention,
            )
            if return_attention:
                temporal_attentions.append(attn_weights)

        x = self.final_norm(x)
        seq_repr = x[:, 0]

        extra = {}
        if return_attention:
            extra["temporal_attentions"] = temporal_attentions
            extra["temporal_cls_attn"] = temporal_attentions[-1][:, :, 0, 1:]
        return seq_repr, extra


class NormalInverseGammaHead(nn.Module):
    """Evidential regression head for uncertainty-aware slope estimation."""

    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 4),
        )

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        gamma, v_raw, alpha_raw, beta_raw = self.net(x).chunk(4, dim=-1)
        v = F.softplus(v_raw) + 1e-6
        alpha = F.softplus(alpha_raw) + 1.0 + 1e-6
        beta = F.softplus(beta_raw) + 1e-6
        return {
            "gamma": gamma.squeeze(-1),
            "v": v.squeeze(-1),
            "alpha": alpha.squeeze(-1),
            "beta": beta.squeeze(-1),
        }


class SpatioTemporalTransformer(nn.Module):
    """
    Main paper model.
    The final model is a hybrid raw-grid + tabular architecture:
    dual-branch spatio-temporal encoding of TD/HVF grids plus a gated adapter for
    derived tabular biomarkers. This addresses the empirical reality that glaucoma
    progression can be captured both by local field topology and by global clinical indices.
    """

    def __init__(
        self,
        embed_dim: int = 128,
        spatial_layers: int = 3,
        temporal_layers: int = 3,
        num_heads: int = 4,
        dropout: float = 0.1,
        visit_stats_dim: int = 7,
        tabular_dim: int = 0,
    ):
        super().__init__()
        self.tabular_dim = tabular_dim
        self.td_encoder = SingleModalitySpatialEncoder(
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_layers=spatial_layers,
            dropout=dropout,
        )
        self.hvf_encoder = SingleModalitySpatialEncoder(
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_layers=spatial_layers,
            dropout=dropout,
        )
        self.fusion = CrossModalGatedFusion(dim=embed_dim, dropout=dropout)
        self.temporal_encoder = TemporalEncoder(
            spatial_dim=embed_dim,
            visit_stats_dim=visit_stats_dim,
            model_dim=embed_dim,
            num_heads=num_heads,
            num_layers=temporal_layers,
            dropout=dropout,
        )
        self.shared = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        if tabular_dim > 0:
            self.tabular_adapter = nn.Sequential(
                nn.LayerNorm(tabular_dim),
                nn.Linear(tabular_dim, embed_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim, embed_dim),
            )
            self.tabular_gate = nn.Sequential(
                nn.Linear(embed_dim * 3, embed_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim, embed_dim),
                nn.Sigmoid(),
            )
            self.tabular_norm = nn.LayerNorm(embed_dim)
        else:
            self.tabular_adapter = None
            self.tabular_gate = None
            self.tabular_norm = None
        self.regression_head = NormalInverseGammaHead(embed_dim, dropout=dropout)
        self.classification_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, 1),
        )

    def forward(self, batch: Dict[str, torch.Tensor], return_attention: bool = False) -> Dict[str, torch.Tensor]:
        td_tokens, td_info = self.td_encoder(
            modality_seq=batch["td_seq"],
            valid_mask_tensor=batch["valid_mask"],
            return_attention=return_attention,
            prefix_name="td",
        )
        hvf_tokens, hvf_info = self.hvf_encoder(
            modality_seq=batch["hvf_seq"],
            valid_mask_tensor=batch["valid_mask"],
            return_attention=return_attention,
            prefix_name="hvf",
        )
        fused_tokens, fusion_info = self.fusion(
            td_tokens=td_tokens,
            hvf_tokens=hvf_tokens,
            return_attention=return_attention,
        )
        seq_repr, temporal_info = self.temporal_encoder(
            spatial_tokens=fused_tokens,
            visit_stats_seq=batch["visit_stats_seq"],
            age_deltas=batch["age_deltas"],
            age_gaps=batch["age_gaps"],
            padding_mask=batch["padding_mask"],
            return_attention=return_attention,
        )
        seq_repr = self.shared(seq_repr)
        if self.tabular_adapter is not None:
            tabular_repr = self.tabular_adapter(batch["tabular_vector"])
            fusion_inputs = torch.cat([seq_repr, tabular_repr, torch.abs(seq_repr - tabular_repr)], dim=-1)
            tabular_gate = self.tabular_gate(fusion_inputs)
            seq_repr = self.tabular_norm(tabular_gate * seq_repr + (1.0 - tabular_gate) * tabular_repr)
        else:
            tabular_gate = None
        nig = self.regression_head(seq_repr)
        fast_logit = self.classification_head(seq_repr).squeeze(-1)

        output = {
            **nig,
            "slope_pred": nig["gamma"],
            "fast_logit": fast_logit,
        }
        if return_attention:
            output.update(td_info)
            output.update(hvf_info)
            output.update(fusion_info)
            output.update(temporal_info)
            if tabular_gate is not None:
                output["tabular_fusion_gate"] = tabular_gate
        return output

    @torch.no_grad()
    def predict_with_uncertainty(
        self,
        batch: Dict[str, torch.Tensor],
        mc_passes: int = 8,
    ) -> Dict[str, torch.Tensor]:
        self.eval()
        base_output = self.forward(batch, return_attention=False)

        aleatoric_var = base_output["beta"] / torch.clamp(base_output["alpha"] - 1.0, min=1e-6)
        epistemic_var = aleatoric_var / torch.clamp(base_output["v"], min=1e-6)
        slope_mean = base_output["slope_pred"]
        fast_prob = torch.sigmoid(base_output["fast_logit"])

        if mc_passes <= 1:
            total_var = aleatoric_var + epistemic_var
            return {
                "slope_mean": slope_mean,
                "slope_std": torch.sqrt(torch.clamp(total_var, min=1e-8)),
                "fast_prob": fast_prob,
                "aleatoric_var": aleatoric_var,
                "epistemic_var": epistemic_var,
            }

        was_training = self.training
        self.train()
        mc_slope = []
        mc_prob = []
        for _ in range(mc_passes):
            mc_output = self.forward(batch, return_attention=False)
            mc_slope.append(mc_output["slope_pred"].unsqueeze(0))
            mc_prob.append(torch.sigmoid(mc_output["fast_logit"]).unsqueeze(0))
        if not was_training:
            self.eval()

        mc_slope = torch.cat(mc_slope, dim=0)
        mc_prob = torch.cat(mc_prob, dim=0)
        mc_var = mc_slope.var(dim=0, unbiased=False)
        total_var = aleatoric_var + epistemic_var + mc_var

        return {
            "slope_mean": mc_slope.mean(dim=0),
            "slope_std": torch.sqrt(torch.clamp(total_var, min=1e-8)),
            "fast_prob": mc_prob.mean(dim=0),
            "aleatoric_var": aleatoric_var,
            "epistemic_var": epistemic_var + mc_var,
        }


class LSTMBaseline(nn.Module):
    """Ablation baseline using flattened per-visit vectors without spatial attention."""

    def __init__(self, input_dim: int, hidden_dim: int = 128, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.query = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads=4, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.reg_head = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Linear(hidden_dim // 2, 1))
        self.cls_head = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Linear(hidden_dim // 2, 1))
        nn.init.normal_(self.query, mean=0.0, std=0.02)

    def forward(self, batch: Dict[str, torch.Tensor], return_attention: bool = False) -> Dict[str, torch.Tensor]:
        lengths = batch["seq_len"].cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            batch["visit_vector_seq"],
            lengths=lengths,
            batch_first=True,
            enforce_sorted=False,
        )
        packed_out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=batch["visit_vector_seq"].shape[1],
        )
        query = self.query.expand(out.shape[0], 1, out.shape[-1])
        pooled, attn = self.attn(
            query,
            out,
            out,
            key_padding_mask=batch["padding_mask"],
            need_weights=return_attention,
            average_attn_weights=False,
        )
        rep = self.norm(pooled.squeeze(1))
        output = {
            "slope_pred": self.reg_head(rep).squeeze(-1),
            "fast_logit": self.cls_head(rep).squeeze(-1),
        }
        if return_attention:
            output["temporal_cls_attn"] = attn[:, :, 0, :]
        return output


class NumericFeatureTokenizer(nn.Module):
    """Feature tokenization for FT-Transformer style tabular modeling."""

    def __init__(self, num_features: int, token_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_features, token_dim) * 0.02)
        self.bias = nn.Parameter(torch.zeros(num_features, token_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


class FTTransformerBaseline(nn.Module):
    """Tabular-only transformer baseline on derived features."""

    def __init__(
        self,
        num_features: int,
        token_dim: int = 96,
        num_layers: int = 3,
        num_heads: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.tokenizer = NumericFeatureTokenizer(num_features=num_features, token_dim=token_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, token_dim))
        self.blocks = nn.ModuleList(
            [TransformerBlock(token_dim, num_heads=num_heads, dropout=dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(token_dim)
        self.reg_head = nn.Sequential(nn.Linear(token_dim, token_dim // 2), nn.GELU(), nn.Linear(token_dim // 2, 1))
        self.cls_head = nn.Sequential(nn.Linear(token_dim, token_dim // 2), nn.GELU(), nn.Linear(token_dim // 2, 1))
        nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

    def forward(self, batch: Dict[str, torch.Tensor], return_attention: bool = False) -> Dict[str, torch.Tensor]:
        x = self.tokenizer(batch["tabular_vector"])
        cls_token = self.cls_token.expand(x.shape[0], 1, x.shape[-1])
        x = torch.cat([cls_token, x], dim=1)
        attentions = []
        for block in self.blocks:
            x, attn = block(x, key_padding_mask=None, need_weights=return_attention)
            if return_attention:
                attentions.append(attn)
        x = self.norm(x)
        rep = x[:, 0]
        output = {
            "slope_pred": self.reg_head(rep).squeeze(-1),
            "fast_logit": self.cls_head(rep).squeeze(-1),
        }
        if return_attention:
            output["tabular_attentions"] = attentions
        return output


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


sample_visit_vector_dim = example_batch["visit_vector_seq"].shape[-1]
sample_tabular_dim = example_batch["tabular_vector"].shape[-1]

st_model = SpatioTemporalTransformer(
    embed_dim=CONFIG.default_embed_dim,
    spatial_layers=3,
    temporal_layers=3,
    num_heads=4,
    dropout=0.15,
    visit_stats_dim=example_batch["visit_stats_seq"].shape[-1],
    tabular_dim=sample_tabular_dim,
)
lstm_model = LSTMBaseline(input_dim=sample_visit_vector_dim)
ft_model = FTTransformerBaseline(num_features=sample_tabular_dim)

for model_name, model in {
    "ST-Transformer": st_model,
    "LSTM baseline": lstm_model,
    "FT-Transformer baseline": ft_model,
}.items():
    n_params = count_trainable_parameters(model)
    print(f"{model_name}: {n_params:,} parameters")
    assert n_params < 5_000_000, f"{model_name} exceeds CPU-friendly size budget"



## Cell 4 - Training Loop and Hyperparameter Search

This cell trains the main ST-Transformer with Optuna and then fits the ablation baselines. The notebook is accelerator-aware for Kaggle, but it keeps the exact same leakage-safe split logic and reproducibility hooks across CPU and GPU runs. The main objective combines validation MAE and AUC so the selected model is balanced across the two paper endpoints.



In [ ]:
def move_batch_to_device(batch: Dict[str, object], device: torch.device) -> Dict[str, object]:
    moved = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            moved[key] = value.to(device)
        else:
            moved[key] = value
    return moved


def safe_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    return float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan


def make_loader(
    dataset: torch.utils.data.Dataset,
    batch_size: int,
    shuffle: bool,
) -> torch.utils.data.DataLoader:
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        collate_fn=collate_vf_sequences,
    )


def maybe_subsample_dataset(
    dataset: torch.utils.data.Dataset,
    fraction: float,
    seed: int,
    min_samples: int = 64,
) -> torch.utils.data.Dataset:
    if fraction >= 1.0 or len(dataset) <= min_samples:
        return dataset
    rng = np.random.RandomState(seed)
    target_size = min(len(dataset), max(min_samples, int(len(dataset) * fraction)))
    indices = rng.choice(len(dataset), size=target_size, replace=False)
    return torch.utils.data.Subset(dataset, indices.tolist())


def evidential_regression_loss(
    y_true: torch.Tensor,
    gamma: torch.Tensor,
    v: torch.Tensor,
    alpha: torch.Tensor,
    beta: torch.Tensor,
    coeff: float = 1e-2,
) -> torch.Tensor:
    """
    Evidential loss from Amini et al. adapted for robust uncertainty-aware MD slope regression.
    """
    two_beta_lambda = 2.0 * beta * (1.0 + v)
    nll = (
        0.5 * torch.log(torch.tensor(math.pi, device=y_true.device) / v)
        - alpha * torch.log(two_beta_lambda)
        + (alpha + 0.5) * torch.log(v * (y_true - gamma) ** 2 + two_beta_lambda)
        + torch.lgamma(alpha)
        - torch.lgamma(alpha + 0.5)
    )
    reg = torch.abs(y_true - gamma) * (2.0 * v + alpha)
    return (nll + coeff * reg).mean()


def compute_loss(
    model_output: Dict[str, torch.Tensor],
    batch: Dict[str, torch.Tensor],
    pos_weight: torch.Tensor,
    cls_weight: float,
    reg_weight: float,
    evidential_coeff: float,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    y_slope = batch["label_slope"]
    y_fast = batch["label_fast"]
    sample_weight = batch.get("sample_weight")
    if sample_weight is None:
        sample_weight = torch.ones_like(y_slope)
    sample_weight = sample_weight / torch.clamp(sample_weight.mean(), min=1e-6)

    if {"gamma", "v", "alpha", "beta"}.issubset(model_output.keys()):
        reg_loss_unweighted = evidential_regression_loss(
            y_true=y_slope,
            gamma=model_output["gamma"],
            v=model_output["v"],
            alpha=model_output["alpha"],
            beta=model_output["beta"],
            coeff=evidential_coeff,
        )
        tail_loss = F.smooth_l1_loss(model_output["gamma"], y_slope, reduction="none")
        reg_loss = 0.70 * reg_loss_unweighted + 0.30 * (tail_loss * sample_weight).mean()
    else:
        reg_loss = (F.smooth_l1_loss(model_output["slope_pred"], y_slope, reduction="none") * sample_weight).mean()

    cls_loss_vec = F.binary_cross_entropy_with_logits(
        model_output["fast_logit"],
        y_fast,
        pos_weight=pos_weight,
        reduction="none",
    )
    cls_loss = (cls_loss_vec * sample_weight).mean()

    total_loss = reg_weight * reg_loss + cls_weight * cls_loss
    return total_loss, {"reg_loss": float(reg_loss.item()), "cls_loss": float(cls_loss.item())}


@torch.no_grad()
def predict_deep_model(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
    use_uncertainty: bool = False,
    mc_passes: int = CONFIG.mc_dropout_passes,
) -> Dict[str, np.ndarray]:
    model.eval()

    records = defaultdict(list)
    for batch in loader:
        batch = move_batch_to_device(batch, DEVICE)

        if use_uncertainty and hasattr(model, "predict_with_uncertainty"):
            output = model.predict_with_uncertainty(batch, mc_passes=mc_passes)
            slope_pred = output["slope_mean"]
            slope_std = output["slope_std"]
            fast_prob = output["fast_prob"]
            records["slope_std"].append(slope_std.cpu().numpy())
            records["aleatoric_var"].append(output["aleatoric_var"].cpu().numpy())
            records["epistemic_var"].append(output["epistemic_var"].cpu().numpy())
        else:
            output = model(batch, return_attention=False)
            slope_pred = output["slope_pred"]
            fast_prob = torch.sigmoid(output["fast_logit"])

        records["y_slope"].append(batch["label_slope"].cpu().numpy())
        records["y_fast"].append(batch["label_fast"].cpu().numpy())
        records["event_time"].append(batch["event_time"].cpu().numpy())
        records["event_observed"].append(batch["event_observed"].cpu().numpy())
        records["baseline_age"].append(batch["baseline_age"].cpu().numpy())
        records["baseline_md"].append(batch["baseline_md"].cpu().numpy())
        records["slope_pred"].append(slope_pred.cpu().numpy())
        records["fast_prob"].append(fast_prob.cpu().numpy())
        records["gender"].extend(batch["gender"])
        records["patient_id"].extend(batch["patient_id"])
        records["eye_uid"].extend(batch["eye_uid"])

    output = {key: np.concatenate(value) if key not in {"gender", "patient_id", "eye_uid"} else np.asarray(value)
              for key, value in records.items()}
    return output


def choose_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    best_threshold = 0.5
    best_f1 = -np.inf
    for threshold in np.linspace(0.1, 0.9, 81):
        y_pred = (y_prob >= threshold).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)
    return best_threshold


def compute_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Version-safe RMSE helper.
    scikit-learn >= 1.6 removed the `squared` argument from `mean_squared_error`,
    so we compute RMSE explicitly from MSE.
    """
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_correlation(x: np.ndarray, y: np.ndarray, method: str = "pearson") -> float:
    if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    if method == "pearson":
        return float(stats.pearsonr(x, y)[0])
    if method == "spearman":
        return float(stats.spearmanr(x, y)[0])
    raise ValueError(f"Unsupported method: {method}")


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for bin_idx in range(n_bins):
        mask = bin_ids == bin_idx
        if not np.any(mask):
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)


def summarize_predictions(preds: Dict[str, np.ndarray], threshold: float) -> Dict[str, float]:
    y_slope = preds["y_slope"]
    slope_pred = preds["slope_pred"]
    y_fast = preds["y_fast"].astype(int)
    fast_prob = preds["fast_prob"]
    fast_pred = (fast_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_fast, fast_pred, labels=[0, 1]).ravel()
    specificity = tn / max(tn + fp, 1)
    npv = tn / max(tn + fn, 1)
    fpr = fp / max(fp + tn, 1)
    fnr = fn / max(fn + tp, 1)

    metrics = {
        "MAE": mean_absolute_error(y_slope, slope_pred),
        "RMSE": compute_rmse(y_slope, slope_pred),
        "MedianAE": median_absolute_error(y_slope, slope_pred),
        "R2": r2_score(y_slope, slope_pred),
        "ExplainedVar": explained_variance_score(y_slope, slope_pred),
        "Bias": float(np.mean(slope_pred - y_slope)),
        "PearsonR": safe_correlation(y_slope, slope_pred, method="pearson"),
        "SpearmanR": safe_correlation(y_slope, slope_pred, method="spearman"),
        "AUC": safe_auc(y_fast, fast_prob),
        "AP": average_precision_score(y_fast, fast_prob),
        "Accuracy": float(np.mean(y_fast == fast_pred)),
        "F1": f1_score(y_fast, fast_pred, zero_division=0),
        "Precision": precision_score(y_fast, fast_pred, zero_division=0),
        "Recall": recall_score(y_fast, fast_pred, zero_division=0),
        "Sensitivity": recall_score(y_fast, fast_pred, zero_division=0),
        "Specificity": specificity,
        "BalancedAcc": balanced_accuracy_score(y_fast, fast_pred),
        "MCC": matthews_corrcoef(y_fast, fast_pred),
        "NPV": npv,
        "FPR": fpr,
        "FNR": fnr,
        "Brier": brier_score_loss(y_fast, fast_prob),
        "LogLoss": log_loss(y_fast, np.clip(fast_prob, 1e-6, 1.0 - 1e-6), labels=[0, 1]),
        "ECE": expected_calibration_error(y_fast, fast_prob),
        # Higher predicted slope should correspond to longer event-free follow-up.
        "C-index": concordance_index(preds["event_time"], slope_pred, preds["event_observed"]),
    }
    return {key: float(value) for key, value in metrics.items()}


def fit_deep_model(
    model: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    lr: float,
    weight_decay: float,
    max_epochs: int,
    patience: int,
    cls_weight: float,
    reg_weight: float,
    evidential_coeff: float,
) -> Tuple[nn.Module, pd.DataFrame]:
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )

    pos_fraction = float(np.mean([sample["label_fast"] for sample in train_samples]))
    pos_weight = torch.tensor((1.0 - pos_fraction) / max(pos_fraction, 1e-6), dtype=torch.float32, device=DEVICE)

    best_state = copy.deepcopy(model.state_dict())
    best_score = np.inf
    best_epoch = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        train_reg_losses = []
        train_cls_losses = []

        for batch in train_loader:
            batch = move_batch_to_device(batch, DEVICE)
            optimizer.zero_grad(set_to_none=True)
            output = model(batch, return_attention=False)
            loss, loss_parts = compute_loss(
                model_output=output,
                batch=batch,
                pos_weight=pos_weight,
                cls_weight=cls_weight,
                reg_weight=reg_weight,
                evidential_coeff=evidential_coeff,
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_losses.append(float(loss.item()))
            train_reg_losses.append(loss_parts["reg_loss"])
            train_cls_losses.append(loss_parts["cls_loss"])

        val_preds = predict_deep_model(model, val_loader, use_uncertainty=False)
        val_threshold = choose_threshold(val_preds["y_fast"], val_preds["fast_prob"])
        val_metrics = summarize_predictions(val_preds, threshold=val_threshold)

        val_composite = val_metrics["MAE"] + (1.0 - val_metrics["AUC"])
        scheduler.step(val_composite)

        history.append(
            {
                "epoch": epoch,
                "train_loss": np.mean(train_losses),
                "train_reg_loss": np.mean(train_reg_losses),
                "train_cls_loss": np.mean(train_cls_losses),
                "val_threshold": val_threshold,
                "val_composite": val_composite,
                **{f"val_{metric}": value for metric, value in val_metrics.items()},
            }
        )

        if val_composite < best_score:
            best_score = val_composite
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
        elif epoch - best_epoch >= patience:
            break

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    return model, history_df


def make_st_model_from_params(params: Dict[str, float]) -> SpatioTemporalTransformer:
    model = SpatioTemporalTransformer(
        embed_dim=int(params["embed_dim"]),
        spatial_layers=int(params["spatial_layers"]),
        temporal_layers=int(params["temporal_layers"]),
        num_heads=int(params["num_heads"]),
        dropout=float(params["dropout"]),
        visit_stats_dim=example_batch["visit_stats_seq"].shape[-1],
        tabular_dim=sample_tabular_dim,
    )
    return model


def objective(trial: optuna.trial.Trial) -> float:
    embed_choices = [64, 96] if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else [96, 128]
    spatial_range = (1, 2) if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else (2, 3)
    temporal_range = (1, 2) if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else (2, 4)
    head_choices = [2, 4] if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else [4, 8]
    batch_choices = [4, 8] if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else [8, 16, 32]

    params = {
        "embed_dim": trial.suggest_categorical("embed_dim", embed_choices),
        "spatial_layers": trial.suggest_int("spatial_layers", spatial_range[0], spatial_range[1]),
        "temporal_layers": trial.suggest_int("temporal_layers", temporal_range[0], temporal_range[1]),
        "num_heads": trial.suggest_categorical("num_heads", head_choices),
        "dropout": trial.suggest_float(
            "dropout",
            0.10,
            0.25 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 0.35,
        ),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 5e-4, log=True),
        "batch_size": trial.suggest_categorical("batch_size", batch_choices),
        "cls_weight": trial.suggest_float("cls_weight", 0.6, 1.4),
        "reg_weight": trial.suggest_float("reg_weight", 0.6, 1.4),
        "evidential_coeff": trial.suggest_float("evidential_coeff", 1e-3, 5e-2, log=True),
    }

    train_dataset_trial = maybe_subsample_dataset(
        train_dataset,
        fraction=CONFIG.objective_train_fraction,
        seed=SEED + trial.number,
    )
    val_dataset_trial = maybe_subsample_dataset(
        val_dataset,
        fraction=CONFIG.objective_val_fraction,
        seed=SEED + 1000 + trial.number,
    )

    train_loader_trial = make_loader(train_dataset_trial, batch_size=int(params["batch_size"]), shuffle=True)
    val_loader_trial = make_loader(val_dataset_trial, batch_size=int(params["batch_size"]), shuffle=False)

    model = make_st_model_from_params(params)
    assert count_trainable_parameters(model) < 5_000_000

    model, history_df = fit_deep_model(
        model=model,
        train_loader=train_loader_trial,
        val_loader=val_loader_trial,
        lr=float(params["lr"]),
        weight_decay=float(params["weight_decay"]),
        max_epochs=CONFIG.optuna_epochs,
        patience=4,
        cls_weight=float(params["cls_weight"]),
        reg_weight=float(params["reg_weight"]),
        evidential_coeff=float(params["evidential_coeff"]),
    )

    best_score = float(history_df["val_composite"].min())
    trial.set_user_attr("best_epoch", int(history_df.loc[history_df["val_composite"].idxmin(), "epoch"]))
    return best_score


sampler = optuna.samplers.TPESampler(seed=SEED)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=4)
study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
study.optimize(
    objective,
    n_trials=CONFIG.optuna_trials,
    timeout=CONFIG.optuna_timeout_seconds,
    show_progress_bar=False,
)

study.trials_dataframe().to_csv(CONFIG.output_dir / "optuna_trials.csv", index=False)

best_params = study.best_trial.params
with open(CONFIG.output_dir / "best_st_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)
display(pd.Series(best_params, name="best_value").to_frame())

best_st_model = make_st_model_from_params(best_params)
best_st_model, st_history = fit_deep_model(
    model=best_st_model,
    train_loader=make_loader(train_dataset, batch_size=int(best_params["batch_size"]), shuffle=True),
    val_loader=make_loader(val_dataset, batch_size=int(best_params["batch_size"]), shuffle=False),
    lr=float(best_params["lr"]),
    weight_decay=float(best_params["weight_decay"]),
    max_epochs=CONFIG.max_epochs,
    patience=CONFIG.patience,
    cls_weight=float(best_params["cls_weight"]),
    reg_weight=float(best_params["reg_weight"]),
    evidential_coeff=float(best_params["evidential_coeff"]),
)

torch.save(best_st_model.state_dict(), CONFIG.output_dir / f"best_st_transformer_{DEVICE.type}.pt")
st_history.to_csv(CONFIG.output_dir / "st_transformer_training_history.csv", index=False)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(st_history["epoch"], st_history["train_loss"], label="Train loss")
ax.plot(st_history["epoch"], st_history["val_composite"], label="Validation composite")
ax.set_title("ST-Transformer Training History")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss / composite score")
ax.legend()
plt.tight_layout()
save_figure(fig, "st_transformer_training_history")
plt.show()


def build_tabular_matrix(samples: Sequence[Dict[str, object]], normalizer: FeatureNormalizer):
    x = np.vstack([normalizer.normalize_tabular(sample["tabular_vector"]) for sample in samples]).astype(np.float32)
    y_reg = np.asarray([sample["label_slope"] for sample in samples], dtype=np.float32)
    y_cls = np.asarray([sample["label_fast"] for sample in samples], dtype=np.int64)
    event_time = np.asarray([sample["event_time"] for sample in samples], dtype=np.float32)
    event_observed = np.asarray([sample["event_observed"] for sample in samples], dtype=np.float32)
    baseline_age = np.asarray([sample["baseline_age"] for sample in samples], dtype=np.float32)
    baseline_md = np.asarray([sample["baseline_md"] for sample in samples], dtype=np.float32)
    sample_weight = np.asarray([sample.get("sample_weight", 1.0) for sample in samples], dtype=np.float32)
    genders = np.asarray([sample["gender"] for sample in samples])
    feature_names = list(samples[0]["tabular_feature_names"])
    return x, y_reg, y_cls, event_time, event_observed, baseline_age, baseline_md, sample_weight, genders, feature_names


X_train, y_train_reg, y_train_cls, _, _, _, _, train_sample_weight, _, tab_feature_names = build_tabular_matrix(train_samples, normalizer)
X_val, y_val_reg, y_val_cls, _, _, val_baseline_age, val_baseline_md, val_sample_weight, val_genders, _ = build_tabular_matrix(val_samples, normalizer)
X_test, y_test_reg, y_test_cls, test_event_time, test_event_observed, test_baseline_age, test_baseline_md, test_sample_weight, test_genders, _ = build_tabular_matrix(
    test_samples,
    normalizer,
)

boosting_reg_estimators = 300 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 700
boosting_cls_estimators = 250 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 600

if CONFIG.train_boosting_baseline and lgb is not None:
    reg_baseline = lgb.LGBMRegressor(
        objective="mae",
        n_estimators=boosting_reg_estimators,
        learning_rate=0.05 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 0.03,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=SEED,
    )
    cls_baseline = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=boosting_cls_estimators,
        learning_rate=0.05 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 0.03,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=SEED,
    )
    reg_baseline.fit(
        X_train,
        y_train_reg,
        sample_weight=train_sample_weight,
        eval_set=[(X_val, y_val_reg)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )
    cls_baseline.fit(
        X_train,
        y_train_cls,
        sample_weight=train_sample_weight,
        eval_set=[(X_val, y_val_cls)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )
    boosting_name = "LightGBM"
elif CONFIG.train_boosting_baseline and xgb is not None:
    reg_baseline = xgb.XGBRegressor(
        n_estimators=boosting_reg_estimators,
        learning_rate=0.05 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 0.03,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="reg:absoluteerror",
        random_state=SEED,
        n_jobs=MAX_CPU_THREADS,
    )
    cls_baseline = xgb.XGBClassifier(
        n_estimators=boosting_cls_estimators,
        learning_rate=0.05 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 0.03,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=SEED,
        n_jobs=MAX_CPU_THREADS,
    )
    reg_baseline.fit(X_train, y_train_reg, sample_weight=train_sample_weight, eval_set=[(X_val, y_val_reg)], verbose=False)
    cls_baseline.fit(X_train, y_train_cls, sample_weight=train_sample_weight, eval_set=[(X_val, y_val_cls)], verbose=False)
    boosting_name = "XGBoost"
elif CONFIG.train_boosting_baseline:
    raise ImportError("Install either LightGBM or XGBoost to run the tabular boosting baseline.")
else:
    reg_baseline = None
    cls_baseline = None
    boosting_name = None

extra_trees_reg = ExtraTreesRegressor(
    n_estimators=300 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 600,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=MAX_CPU_THREADS,
)
extra_trees_cls = ExtraTreesClassifier(
    n_estimators=300 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 600,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=MAX_CPU_THREADS,
)
extra_trees_reg.fit(X_train, y_train_reg, sample_weight=train_sample_weight)
extra_trees_cls.fit(X_train, y_train_cls, sample_weight=train_sample_weight)

hgb_reg = HistGradientBoostingRegressor(
    max_iter=250 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 400,
    learning_rate=0.05,
    max_depth=6,
    random_state=SEED,
)
hgb_cls = HistGradientBoostingClassifier(
    max_iter=250 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 400,
    learning_rate=0.05,
    max_depth=6,
    random_state=SEED,
)
hgb_reg.fit(X_train, y_train_reg, sample_weight=train_sample_weight)
hgb_cls.fit(X_train, y_train_cls, sample_weight=train_sample_weight)

catboost_reg = None
catboost_cls = None
if CONFIG.train_catboost_baseline and CatBoostRegressor is not None and CatBoostClassifier is not None:
    catboost_reg = CatBoostRegressor(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=500 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 900,
        learning_rate=0.05 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 0.03,
        depth=6,
        random_seed=SEED,
        verbose=False,
    )
    catboost_cls = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=450 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 800,
        learning_rate=0.05 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 0.03,
        depth=6,
        random_seed=SEED,
        auto_class_weights="Balanced",
        verbose=False,
    )
    catboost_reg.fit(X_train, y_train_reg, sample_weight=train_sample_weight, eval_set=(X_val, y_val_reg), use_best_model=True)
    catboost_cls.fit(X_train, y_train_cls, sample_weight=train_sample_weight, eval_set=(X_val, y_val_cls), use_best_model=True)


lstm_baseline = None
ft_baseline = None

if CONFIG.train_deep_baselines:
    lstm_baseline = LSTMBaseline(input_dim=sample_visit_vector_dim, hidden_dim=96, num_layers=2, dropout=0.2)
    lstm_baseline, lstm_history = fit_deep_model(
        model=lstm_baseline,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=1e-3,
        weight_decay=1e-5,
        max_epochs=20 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 35,
        patience=4 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 6,
        cls_weight=1.0,
        reg_weight=1.0,
        evidential_coeff=0.0,
    )

    ft_baseline = FTTransformerBaseline(num_features=sample_tabular_dim, token_dim=64, num_layers=2, num_heads=4, dropout=0.1)
    ft_baseline, ft_history = fit_deep_model(
        model=ft_baseline,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=8e-4,
        weight_decay=1e-5,
        max_epochs=20 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 35,
        patience=4 if CONFIG.compute_profile in LOW_RESOURCE_PROFILES else 6,
        cls_weight=1.0,
        reg_weight=1.0,
        evidential_coeff=0.0,
    )


predictions = {}

predictions["ST-Transformer"] = {
    "val": predict_deep_model(best_st_model, val_loader, use_uncertainty=True, mc_passes=CONFIG.mc_dropout_passes),
    "test": predict_deep_model(best_st_model, test_loader, use_uncertainty=True, mc_passes=CONFIG.mc_dropout_passes),
}

if lstm_baseline is not None:
    predictions["LSTM"] = {
        "val": predict_deep_model(lstm_baseline, val_loader, use_uncertainty=False),
        "test": predict_deep_model(lstm_baseline, test_loader, use_uncertainty=False),
    }

if ft_baseline is not None:
    predictions["FT-Transformer"] = {
        "val": predict_deep_model(ft_baseline, val_loader, use_uncertainty=False),
        "test": predict_deep_model(ft_baseline, test_loader, use_uncertainty=False),
    }

predictions["ExtraTrees"] = {
    "val": {
        "y_slope": y_val_reg,
        "y_fast": y_val_cls,
        "event_time": np.asarray([sample["event_time"] for sample in val_samples], dtype=np.float32),
        "event_observed": np.asarray([sample["event_observed"] for sample in val_samples], dtype=np.float32),
        "baseline_age": val_baseline_age,
        "baseline_md": val_baseline_md,
        "slope_pred": extra_trees_reg.predict(X_val),
        "fast_prob": extra_trees_cls.predict_proba(X_val)[:, 1],
        "gender": np.asarray([sample["gender"] for sample in val_samples]),
        "patient_id": np.asarray([sample["patient_id"] for sample in val_samples]),
        "eye_uid": np.asarray([sample["eye_uid"] for sample in val_samples]),
    },
    "test": {
        "y_slope": y_test_reg,
        "y_fast": y_test_cls,
        "event_time": test_event_time,
        "event_observed": test_event_observed,
        "baseline_age": test_baseline_age,
        "baseline_md": test_baseline_md,
        "slope_pred": extra_trees_reg.predict(X_test),
        "fast_prob": extra_trees_cls.predict_proba(X_test)[:, 1],
        "gender": test_genders,
        "patient_id": np.asarray([sample["patient_id"] for sample in test_samples]),
        "eye_uid": np.asarray([sample["eye_uid"] for sample in test_samples]),
    },
}

predictions["HistGB"] = {
    "val": {
        "y_slope": y_val_reg,
        "y_fast": y_val_cls,
        "event_time": np.asarray([sample["event_time"] for sample in val_samples], dtype=np.float32),
        "event_observed": np.asarray([sample["event_observed"] for sample in val_samples], dtype=np.float32),
        "baseline_age": val_baseline_age,
        "baseline_md": val_baseline_md,
        "slope_pred": hgb_reg.predict(X_val),
        "fast_prob": hgb_cls.predict_proba(X_val)[:, 1],
        "gender": np.asarray([sample["gender"] for sample in val_samples]),
        "patient_id": np.asarray([sample["patient_id"] for sample in val_samples]),
        "eye_uid": np.asarray([sample["eye_uid"] for sample in val_samples]),
    },
    "test": {
        "y_slope": y_test_reg,
        "y_fast": y_test_cls,
        "event_time": test_event_time,
        "event_observed": test_event_observed,
        "baseline_age": test_baseline_age,
        "baseline_md": test_baseline_md,
        "slope_pred": hgb_reg.predict(X_test),
        "fast_prob": hgb_cls.predict_proba(X_test)[:, 1],
        "gender": test_genders,
        "patient_id": np.asarray([sample["patient_id"] for sample in test_samples]),
        "eye_uid": np.asarray([sample["eye_uid"] for sample in test_samples]),
    },
}

if catboost_reg is not None and catboost_cls is not None:
    predictions["CatBoost"] = {
        "val": {
            "y_slope": y_val_reg,
            "y_fast": y_val_cls,
            "event_time": np.asarray([sample["event_time"] for sample in val_samples], dtype=np.float32),
            "event_observed": np.asarray([sample["event_observed"] for sample in val_samples], dtype=np.float32),
            "baseline_age": val_baseline_age,
            "baseline_md": val_baseline_md,
            "slope_pred": catboost_reg.predict(X_val),
            "fast_prob": catboost_cls.predict_proba(X_val)[:, 1],
            "gender": np.asarray([sample["gender"] for sample in val_samples]),
            "patient_id": np.asarray([sample["patient_id"] for sample in val_samples]),
            "eye_uid": np.asarray([sample["eye_uid"] for sample in val_samples]),
        },
        "test": {
            "y_slope": y_test_reg,
            "y_fast": y_test_cls,
            "event_time": test_event_time,
            "event_observed": test_event_observed,
            "baseline_age": test_baseline_age,
            "baseline_md": test_baseline_md,
            "slope_pred": catboost_reg.predict(X_test),
            "fast_prob": catboost_cls.predict_proba(X_test)[:, 1],
            "gender": test_genders,
            "patient_id": np.asarray([sample["patient_id"] for sample in test_samples]),
            "eye_uid": np.asarray([sample["eye_uid"] for sample in test_samples]),
        },
    }

if reg_baseline is not None and cls_baseline is not None and boosting_name is not None:
    val_boosting_prob = cls_baseline.predict_proba(X_val)[:, 1]
    test_boosting_prob = cls_baseline.predict_proba(X_test)[:, 1]

    predictions[boosting_name] = {
        "val": {
            "y_slope": y_val_reg,
            "y_fast": y_val_cls,
            "event_time": np.asarray([sample["event_time"] for sample in val_samples], dtype=np.float32),
            "event_observed": np.asarray([sample["event_observed"] for sample in val_samples], dtype=np.float32),
            "baseline_age": val_baseline_age,
            "baseline_md": val_baseline_md,
            "slope_pred": reg_baseline.predict(X_val),
            "fast_prob": val_boosting_prob,
            "gender": np.asarray([sample["gender"] for sample in val_samples]),
            "patient_id": np.asarray([sample["patient_id"] for sample in val_samples]),
            "eye_uid": np.asarray([sample["eye_uid"] for sample in val_samples]),
        },
        "test": {
            "y_slope": y_test_reg,
            "y_fast": y_test_cls,
            "event_time": test_event_time,
            "event_observed": test_event_observed,
            "baseline_age": test_baseline_age,
            "baseline_md": test_baseline_md,
            "slope_pred": reg_baseline.predict(X_test),
            "fast_prob": test_boosting_prob,
            "gender": test_genders,
            "patient_id": np.asarray([sample["patient_id"] for sample in test_samples]),
            "eye_uid": np.asarray([sample["eye_uid"] for sample in test_samples]),
        },
    }

stacking_model_names = ["ST-Transformer", "ExtraTrees", "HistGB"]
if boosting_name is not None:
    stacking_model_names.insert(1, boosting_name)
if "CatBoost" in predictions:
    stacking_model_names.append("CatBoost")

if len(stacking_model_names) >= 3:
    ensemble_reg_features_val = np.column_stack(
        [predictions[model_name]["val"]["slope_pred"] for model_name in stacking_model_names]
    )
    ensemble_reg_features_test = np.column_stack(
        [predictions[model_name]["test"]["slope_pred"] for model_name in stacking_model_names]
    )
    ensemble_cls_features_val = np.column_stack(
        [predictions[model_name]["val"]["fast_prob"] for model_name in stacking_model_names]
    )
    ensemble_cls_features_test = np.column_stack(
        [predictions[model_name]["test"]["fast_prob"] for model_name in stacking_model_names]
    )

    reg_ensemble = Ridge(alpha=1.0, random_state=SEED)
    cls_ensemble = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    reg_ensemble.fit(ensemble_reg_features_val, y_val_reg)
    cls_ensemble.fit(ensemble_cls_features_val, y_val_cls)

    predictions["Hybrid Ensemble"] = {
        "val": {
            "y_slope": y_val_reg,
            "y_fast": y_val_cls,
            "event_time": np.asarray([sample["event_time"] for sample in val_samples], dtype=np.float32),
            "event_observed": np.asarray([sample["event_observed"] for sample in val_samples], dtype=np.float32),
            "baseline_age": val_baseline_age,
            "baseline_md": val_baseline_md,
            "slope_pred": reg_ensemble.predict(ensemble_reg_features_val),
            "fast_prob": cls_ensemble.predict_proba(ensemble_cls_features_val)[:, 1],
            "gender": np.asarray([sample["gender"] for sample in val_samples]),
            "patient_id": np.asarray([sample["patient_id"] for sample in val_samples]),
            "eye_uid": np.asarray([sample["eye_uid"] for sample in val_samples]),
        },
        "test": {
            "y_slope": y_test_reg,
            "y_fast": y_test_cls,
            "event_time": test_event_time,
            "event_observed": test_event_observed,
            "baseline_age": test_baseline_age,
            "baseline_md": test_baseline_md,
            "slope_pred": reg_ensemble.predict(ensemble_reg_features_test),
            "fast_prob": cls_ensemble.predict_proba(ensemble_cls_features_test)[:, 1],
            "gender": test_genders,
            "patient_id": np.asarray([sample["patient_id"] for sample in test_samples]),
            "eye_uid": np.asarray([sample["eye_uid"] for sample in test_samples]),
        },
    }


def optimize_simplex_weights(
    val_matrix: np.ndarray,
    y_true: np.ndarray,
    objective: str,
    n_draws: int = 5000,
    seed: int = SEED,
) -> np.ndarray:
    """
    Robust validation-tuned ensemble weights.
    Nonnegative simplex weights are less fragile than unconstrained meta-models
    when the calibration split is intentionally small and patient-independent.
    """
    rng = np.random.RandomState(seed)
    n_models = val_matrix.shape[1]
    candidates = [np.ones(n_models, dtype=np.float64) / n_models]
    candidates.extend(np.eye(n_models, dtype=np.float64))
    candidates.extend(rng.dirichlet(np.ones(n_models), size=n_draws))

    best_score = np.inf
    best_weights = candidates[0]
    for weights in candidates:
        pred = val_matrix @ weights
        if objective == "mae":
            score = mean_absolute_error(y_true, pred)
        elif objective == "log_loss":
            score = log_loss(y_true, np.clip(pred, 1e-6, 1.0 - 1e-6), labels=[0, 1])
        else:
            raise ValueError(f"Unsupported objective: {objective}")
        if score < best_score:
            best_score = score
            best_weights = weights
    return np.asarray(best_weights, dtype=np.float32)


ensemble_candidate_names = [
    name
    for name in ["ST-Transformer", "FT-Transformer", "CatBoost", boosting_name, "ExtraTrees", "HistGB"]
    if name is not None and name in predictions
]
if len(ensemble_candidate_names) >= 3:
    constrained_reg_val = np.column_stack(
        [predictions[name]["val"]["slope_pred"] for name in ensemble_candidate_names]
    )
    constrained_reg_test = np.column_stack(
        [predictions[name]["test"]["slope_pred"] for name in ensemble_candidate_names]
    )
    constrained_cls_val = np.column_stack(
        [predictions[name]["val"]["fast_prob"] for name in ensemble_candidate_names]
    )
    constrained_cls_test = np.column_stack(
        [predictions[name]["test"]["fast_prob"] for name in ensemble_candidate_names]
    )

    constrained_reg_weights = optimize_simplex_weights(
        constrained_reg_val,
        y_val_reg,
        objective="mae",
        seed=SEED + 11,
    )
    constrained_cls_weights = optimize_simplex_weights(
        constrained_cls_val,
        y_val_cls,
        objective="log_loss",
        seed=SEED + 23,
    )
    pd.DataFrame(
        {
            "Model": ensemble_candidate_names,
            "regression_weight": constrained_reg_weights,
            "classification_weight": constrained_cls_weights,
        }
    ).to_csv(CONFIG.output_dir / "constrained_ensemble_weights.csv", index=False)

    predictions["Constrained Ensemble"] = {
        "val": {
            "y_slope": y_val_reg,
            "y_fast": y_val_cls,
            "event_time": np.asarray([sample["event_time"] for sample in val_samples], dtype=np.float32),
            "event_observed": np.asarray([sample["event_observed"] for sample in val_samples], dtype=np.float32),
            "baseline_age": val_baseline_age,
            "baseline_md": val_baseline_md,
            "slope_pred": constrained_reg_val @ constrained_reg_weights,
            "fast_prob": np.clip(constrained_cls_val @ constrained_cls_weights, 1e-6, 1.0 - 1e-6),
            "gender": val_genders,
            "patient_id": np.asarray([sample["patient_id"] for sample in val_samples]),
            "eye_uid": np.asarray([sample["eye_uid"] for sample in val_samples]),
        },
        "test": {
            "y_slope": y_test_reg,
            "y_fast": y_test_cls,
            "event_time": test_event_time,
            "event_observed": test_event_observed,
            "baseline_age": test_baseline_age,
            "baseline_md": test_baseline_md,
            "slope_pred": constrained_reg_test @ constrained_reg_weights,
            "fast_prob": np.clip(constrained_cls_test @ constrained_cls_weights, 1e-6, 1.0 - 1e-6),
            "gender": test_genders,
            "patient_id": np.asarray([sample["patient_id"] for sample in test_samples]),
            "eye_uid": np.asarray([sample["eye_uid"] for sample in test_samples]),
        },
    }

decision_thresholds = {
    model_name: choose_threshold(bundle["val"]["y_fast"], bundle["val"]["fast_prob"])
    for model_name, bundle in predictions.items()
}
decision_thresholds



## Cell 5 - Evaluation and Clinical Metrics

This cell quantifies primary and secondary endpoints, compares all ablations, audits fairness by sex and age, and exports a publication-ready LaTeX table. The event-based component uses a clinically interpretable first significant MD drop endpoint for concordance analysis and Kaplan-Meier style visualization.



In [ ]:
def decision_curve_dataframe(y_true: np.ndarray, y_prob: np.ndarray) -> pd.DataFrame:
    rows = []
    prevalence = y_true.mean()
    for threshold in np.linspace(0.05, 0.95, 19):
        pred_positive = y_prob >= threshold
        tp = np.sum((pred_positive == 1) & (y_true == 1))
        fp = np.sum((pred_positive == 1) & (y_true == 0))
        n = len(y_true)
        net_benefit = (tp / n) - (fp / n) * (threshold / (1.0 - threshold))
        treat_all = prevalence - (1.0 - prevalence) * (threshold / (1.0 - threshold))
        rows.append(
            {
                "threshold": threshold,
                "net_benefit_model": net_benefit,
                "net_benefit_treat_all": treat_all,
                "net_benefit_treat_none": 0.0,
            }
    )
    return pd.DataFrame(rows)


def clone_prediction_bundle(bundle: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    cloned = {}
    for key, value in bundle.items():
        if isinstance(value, np.ndarray):
            cloned[key] = value.copy()
        else:
            cloned[key] = copy.deepcopy(value)
    return cloned


def clone_prediction_dict(pred_dict: Dict[str, Dict[str, Dict[str, np.ndarray]]]) -> Dict[str, Dict[str, Dict[str, np.ndarray]]]:
    return {
        model_name: {split_name: clone_prediction_bundle(split_bundle) for split_name, split_bundle in model_bundle.items()}
        for model_name, model_bundle in pred_dict.items()
    }


def fit_regression_recalibrator(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, float]:
    if len(np.unique(y_pred)) < 3:
        return 1.0, 0.0
    slope, intercept = np.polyfit(y_pred, y_true, deg=1)
    if not np.isfinite(slope) or not np.isfinite(intercept):
        return 1.0, 0.0
    return float(slope), float(intercept)


def apply_regression_recalibration(bundle: Dict[str, np.ndarray], slope: float, intercept: float) -> Dict[str, np.ndarray]:
    calibrated = clone_prediction_bundle(bundle)
    calibrated["slope_pred_raw"] = calibrated["slope_pred"].copy()
    calibrated["slope_pred"] = intercept + slope * calibrated["slope_pred"]
    return calibrated


def fit_probability_calibrator(y_true: np.ndarray, y_prob: np.ndarray):
    if len(np.unique(y_true)) < 2 or len(np.unique(y_prob)) < 3:
        return None
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(y_prob, y_true)
    return calibrator


def apply_probability_calibration(bundle: Dict[str, np.ndarray], calibrator) -> Dict[str, np.ndarray]:
    calibrated = clone_prediction_bundle(bundle)
    calibrated["fast_prob_raw"] = calibrated["fast_prob"].copy()
    calibrated["fast_prob"] = np.clip(calibrator.predict(calibrated["fast_prob"]), 0.0, 1.0)
    return calibrated


def bootstrap_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    metric_fn,
    n_boot: int = CONFIG.bootstrap_replicates,
    seed: int = SEED,
) -> Tuple[float, float]:
    rng = np.random.RandomState(seed)
    values = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        y_true_boot = y_true[idx]
        y_pred_boot = y_pred[idx]
        try:
            value = metric_fn(y_true_boot, y_pred_boot)
        except Exception:
            continue
        if np.isfinite(value):
            values.append(value)
    if len(values) == 0:
        return (np.nan, np.nan)
    return (float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5)))


def bootstrap_prediction_metric_ci(
    preds: Dict[str, np.ndarray],
    threshold: float,
    metric_name: str,
    n_boot: int = CONFIG.bootstrap_replicates,
    seed: int = SEED,
) -> Tuple[float, float]:
    rng = np.random.RandomState(seed)
    n = len(preds["y_fast"])
    values = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        boot_preds = {
            key: value[idx] if isinstance(value, np.ndarray) and len(value) == n else value
            for key, value in preds.items()
        }
        try:
            value = summarize_predictions(boot_preds, threshold=threshold)[metric_name]
        except Exception:
            continue
        if np.isfinite(value):
            values.append(value)
    if len(values) == 0:
        return (np.nan, np.nan)
    return (float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5)))


def format_ci(ci: Tuple[float, float]) -> str:
    return f"[{ci[0]:.3f}, {ci[1]:.3f}]"


raw_predictions = clone_prediction_dict(predictions)
report_predictions = clone_prediction_dict(predictions)
calibration_rows = []

for model_name, bundle in raw_predictions.items():
    calibrated_model_bundle = {"val": clone_prediction_bundle(bundle["val"]), "test": clone_prediction_bundle(bundle["test"])}
    reg_slope, reg_intercept = 1.0, 0.0
    prob_calibrator = None

    if CONFIG.calibrate_regression:
        reg_slope, reg_intercept = fit_regression_recalibrator(bundle["val"]["y_slope"], bundle["val"]["slope_pred"])
        calibrated_model_bundle["val"] = apply_regression_recalibration(bundle["val"], reg_slope, reg_intercept)
        calibrated_model_bundle["test"] = apply_regression_recalibration(bundle["test"], reg_slope, reg_intercept)

    if CONFIG.calibrate_probabilities:
        prob_calibrator = fit_probability_calibrator(
            calibrated_model_bundle["val"]["y_fast"],
            calibrated_model_bundle["val"]["fast_prob"],
        )
        if prob_calibrator is not None:
            calibrated_model_bundle["val"] = apply_probability_calibration(calibrated_model_bundle["val"], prob_calibrator)
            calibrated_model_bundle["test"] = apply_probability_calibration(calibrated_model_bundle["test"], prob_calibrator)

    report_predictions[model_name] = calibrated_model_bundle
    calibration_rows.append(
        {
            "Model": model_name,
            "regression_calibration_slope": reg_slope,
            "regression_calibration_intercept": reg_intercept,
            "val_brier_raw": brier_score_loss(bundle["val"]["y_fast"], np.clip(bundle["val"]["fast_prob"], 1e-6, 1.0 - 1e-6)),
            "val_brier_calibrated": brier_score_loss(
                calibrated_model_bundle["val"]["y_fast"],
                np.clip(calibrated_model_bundle["val"]["fast_prob"], 1e-6, 1.0 - 1e-6),
            ),
            "val_mae_raw": mean_absolute_error(bundle["val"]["y_slope"], bundle["val"]["slope_pred"]),
            "val_mae_calibrated": mean_absolute_error(
                calibrated_model_bundle["val"]["y_slope"],
                calibrated_model_bundle["val"]["slope_pred"],
            ),
        }
    )

calibration_df = pd.DataFrame(calibration_rows).sort_values("val_mae_calibrated")
calibration_df.to_csv(CONFIG.output_dir / "posthoc_calibration_summary.csv", index=False)
display(calibration_df.round(4))

predictions = report_predictions
decision_thresholds = {
    model_name: choose_threshold(bundle["val"]["y_fast"], bundle["val"]["fast_prob"])
    for model_name, bundle in predictions.items()
}

results_rows = []
per_model_errors = {}
ci_rows = []

for model_name, bundle in predictions.items():
    threshold = decision_thresholds[model_name]
    metrics = summarize_predictions(bundle["test"], threshold=threshold)
    results_rows.append({"Model": model_name, **metrics})
    per_model_errors[model_name] = {
        "abs_error": np.abs(bundle["test"]["y_slope"] - bundle["test"]["slope_pred"]),
        "brier_component": (bundle["test"]["y_fast"] - bundle["test"]["fast_prob"]) ** 2,
    }
    mae_ci = bootstrap_ci(bundle["test"]["y_slope"], bundle["test"]["slope_pred"], mean_absolute_error)
    auc_ci = bootstrap_ci(bundle["test"]["y_fast"], bundle["test"]["fast_prob"], safe_auc)
    ap_ci = bootstrap_ci(bundle["test"]["y_fast"], bundle["test"]["fast_prob"], average_precision_score)
    cindex_ci = bootstrap_ci(
        bundle["test"]["event_time"],
        np.column_stack([bundle["test"]["slope_pred"], bundle["test"]["event_observed"]]),
        lambda durations, score_and_event: concordance_index(
            durations,
            score_and_event[:, 0],
            score_and_event[:, 1],
        ),
    )
    metric_ci_map = {
        metric_name: bootstrap_prediction_metric_ci(bundle["test"], threshold, metric_name)
        for metric_name in [
            "Accuracy",
            "F1",
            "Precision",
            "Recall",
            "Sensitivity",
            "Specificity",
            "BalancedAcc",
            "Brier",
            "ECE",
        ]
    }
    ci_rows.append(
        {
            "Model": model_name,
            "MAE_CI_low": mae_ci[0],
            "MAE_CI_high": mae_ci[1],
            "MAE_95CI": format_ci(mae_ci),
            "AUC_CI_low": auc_ci[0],
            "AUC_CI_high": auc_ci[1],
            "AUC_95CI": format_ci(auc_ci),
            "AP_CI_low": ap_ci[0],
            "AP_CI_high": ap_ci[1],
            "AP_95CI": format_ci(ap_ci),
            "CINDEX_CI_low": cindex_ci[0],
            "CINDEX_CI_high": cindex_ci[1],
            "C-index_95CI": format_ci(cindex_ci),
            **{f"{name}_95CI": format_ci(ci) for name, ci in metric_ci_map.items()},
        }
    )

results_df_full = pd.DataFrame(results_rows)
ci_df = pd.DataFrame(ci_rows)
results_df = results_df_full.merge(ci_df, on="Model", how="left")
results_df["Rank_MAE"] = results_df["MAE"].rank(method="min", ascending=True)
results_df["Rank_AUC"] = results_df["AUC"].rank(method="min", ascending=False)
results_df["Rank_AP"] = results_df["AP"].rank(method="min", ascending=False)
results_df["Rank_C-index"] = results_df["C-index"].rank(method="min", ascending=False)
results_df["Rank_Brier"] = results_df["Brier"].rank(method="min", ascending=True)
results_df["Rank_ECE"] = results_df["ECE"].rank(method="min", ascending=True)
results_df["CompositeRank"] = (
    results_df[["Rank_MAE", "Rank_AUC", "Rank_AP", "Rank_C-index", "Rank_Brier", "Rank_ECE"]].mean(axis=1)
)
results_df = results_df.sort_values(by=["CompositeRank", "AUC", "MAE"], ascending=[True, False, True]).reset_index(drop=True)
display(results_df.round(3))

best_overall_model = str(results_df.iloc[0]["Model"])
top_model_names = results_df["Model"].head(min(5, len(results_df))).tolist()

primary_columns = [
    "Model",
    "CompositeRank",
    "MAE",
    "MAE_95CI",
    "RMSE",
    "MedianAE",
    "PearsonR",
    "AUC",
    "AUC_95CI",
    "AP",
    "AP_95CI",
    "Accuracy",
    "Accuracy_95CI",
    "F1",
    "F1_95CI",
    "Precision",
    "Precision_95CI",
    "Sensitivity",
    "Sensitivity_95CI",
    "Specificity",
    "Specificity_95CI",
    "BalancedAcc",
    "BalancedAcc_95CI",
    "MCC",
    "Brier",
    "Brier_95CI",
    "LogLoss",
    "ECE",
    "ECE_95CI",
    "C-index",
    "C-index_95CI",
]
latex_table = results_df[primary_columns].round(3).to_latex(
    index=False,
    escape=False,
    caption="Calibrated ablation study on the UWHVF test set with grouped patient-level hold-out evaluation.",
)
with open(CONFIG.output_dir / "ablation_results.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)

print(latex_table)


reference_model = best_overall_model
stat_rows = []
for model_name in results_df["Model"]:
    if model_name == reference_model:
        stat_rows.append({"Model": model_name, "p_abs_error_vs_reference": np.nan, "p_brier_vs_reference": np.nan})
        continue

    try:
        p_abs = stats.wilcoxon(
            per_model_errors[reference_model]["abs_error"],
            per_model_errors[model_name]["abs_error"],
            alternative="less",
        ).pvalue
    except ValueError:
        p_abs = stats.ttest_rel(
            per_model_errors[reference_model]["abs_error"],
            per_model_errors[model_name]["abs_error"],
            alternative="less",
        ).pvalue

    try:
        p_brier = stats.wilcoxon(
            per_model_errors[reference_model]["brier_component"],
            per_model_errors[model_name]["brier_component"],
            alternative="less",
        ).pvalue
    except ValueError:
        p_brier = stats.ttest_rel(
            per_model_errors[reference_model]["brier_component"],
            per_model_errors[model_name]["brier_component"],
            alternative="less",
        ).pvalue

    stat_rows.append({"Model": model_name, "p_abs_error_vs_reference": p_abs, "p_brier_vs_reference": p_brier})

pvalue_df = pd.DataFrame(stat_rows)
display(pvalue_df)


st_test = predictions["ST-Transformer"]["test"]
best_test = predictions[best_overall_model]["test"]
best_threshold = decision_thresholds[best_overall_model]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ROC
for model_name in top_model_names:
    bundle = predictions[model_name]
    fpr, tpr, _ = roc_curve(bundle["test"]["y_fast"], bundle["test"]["fast_prob"])
    axes[0, 0].plot(
        fpr,
        tpr,
        lw=2.5,
        color=MODEL_COLORS.get(model_name),
        label=f"{model_name} (AUC={safe_auc(bundle['test']['y_fast'], bundle['test']['fast_prob']):.3f})",
    )
axes[0, 0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0, 0].set_title("ROC Curves for Fast Progressor Classification")
axes[0, 0].set_xlabel("False Positive Rate")
axes[0, 0].set_ylabel("True Positive Rate")
axes[0, 0].legend(fontsize=10)

# Precision-recall
for model_name in top_model_names:
    bundle = predictions[model_name]
    precision, recall, _ = precision_recall_curve(bundle["test"]["y_fast"], bundle["test"]["fast_prob"])
    ap = average_precision_score(bundle["test"]["y_fast"], bundle["test"]["fast_prob"])
    axes[0, 1].plot(
        recall,
        precision,
        lw=2.5,
        color=MODEL_COLORS.get(model_name),
        label=f"{model_name} (AP={ap:.3f})",
    )
axes[0, 1].set_title("Precision-Recall Curves")
axes[0, 1].set_xlabel("Recall")
axes[0, 1].set_ylabel("Precision")
axes[0, 1].legend(fontsize=10)

# Calibration
for model_name in top_model_names[:4]:
    CalibrationDisplay.from_predictions(
        predictions[model_name]["test"]["y_fast"],
        predictions[model_name]["test"]["fast_prob"],
        n_bins=8,
        strategy="quantile",
        ax=axes[1, 0],
        name=model_name,
    )
axes[1, 0].set_title("Post-hoc Calibrated Reliability Curves")

# Decision curve
for model_name in top_model_names[:4]:
    dca_df = decision_curve_dataframe(predictions[model_name]["test"]["y_fast"], predictions[model_name]["test"]["fast_prob"])
    axes[1, 1].plot(
        dca_df["threshold"],
        dca_df["net_benefit_model"],
        lw=2.2,
        color=MODEL_COLORS.get(model_name),
        label=model_name,
    )
dca_df = decision_curve_dataframe(best_test["y_fast"], best_test["fast_prob"])
axes[1, 1].plot(dca_df["threshold"], dca_df["net_benefit_treat_all"], linestyle="--", label="Treat all")
axes[1, 1].plot(dca_df["threshold"], dca_df["net_benefit_treat_none"], linestyle=":", label="Treat none")
axes[1, 1].set_title("Decision Curve Analysis")
axes[1, 1].set_xlabel("Threshold probability")
axes[1, 1].set_ylabel("Net benefit")
axes[1, 1].legend()

plt.tight_layout()
save_figure(fig, "clinical_evaluation_panel")
plt.show()


risk_scores = -best_test["slope_pred"]
risk_group_codes = pd.qcut(
    risk_scores,
    q=min(3, len(np.unique(np.round(risk_scores, 6)))),
    labels=False,
    duplicates="drop",
)
risk_groups = pd.Series(risk_group_codes).map({0: "Low risk", 1: "Intermediate risk", 2: "High risk"}).to_numpy()

fig, ax = plt.subplots(1, 1, figsize=(9, 6))
kmf = KaplanMeierFitter()
for label in [item for item in ["Low risk", "Intermediate risk", "High risk"] if item in set(risk_groups)]:
    mask = risk_groups == label
    kmf.fit(
        durations=best_test["event_time"][mask],
        event_observed=best_test["event_observed"][mask],
        label=label,
    )
    kmf.plot_survival_function(ax=ax, ci_show=False)
ax.set_title(f"Event-Free Survival by Predicted Risk Tertile ({best_overall_model})")
ax.set_xlabel("Years from first future visit")
ax.set_ylabel("Event-free probability")
plt.tight_layout()
save_figure(fig, "event_free_survival_risk_tertiles")
plt.show()


subgroup_rows = []
for model_name in results_df["Model"].head(min(3, len(results_df))):
    model_test = predictions[model_name]["test"]
    age_bin_codes = pd.qcut(
        model_test["baseline_age"],
        q=min(4, len(np.unique(np.round(model_test["baseline_age"], 6)))),
        labels=False,
        duplicates="drop",
    )
    age_bins = pd.Series(age_bin_codes).map({0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4"}).astype(str)
    severity_bin_codes = pd.qcut(
        model_test["baseline_md"],
        q=min(4, len(np.unique(np.round(model_test["baseline_md"], 6)))),
        labels=False,
        duplicates="drop",
    )
    severity_bins = pd.Series(severity_bin_codes).map({0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4"}).astype(str)
    for subgroup_name, subgroup_values in {
        "gender": np.unique(model_test["gender"]),
        "age_quartile": np.unique(age_bins),
        "baseline_md_quartile": np.unique(severity_bins),
    }.items():
        for subgroup_value in subgroup_values:
            if subgroup_name == "gender":
                mask = model_test["gender"] == subgroup_value
            elif subgroup_name == "age_quartile":
                mask = age_bins == subgroup_value
            else:
                mask = severity_bins == subgroup_value
            if mask.sum() < 20:
                continue
            subgroup_rows.append(
                {
                    "model": model_name,
                    "group": subgroup_name,
                    "value": subgroup_value,
                    "n": int(mask.sum()),
                    "MAE": mean_absolute_error(model_test["y_slope"][mask], model_test["slope_pred"][mask]),
                    "AUC": safe_auc(model_test["y_fast"][mask], model_test["fast_prob"][mask]),
                    "Fast prevalence": float(model_test["y_fast"][mask].mean()),
                }
            )

fairness_df = pd.DataFrame(subgroup_rows)
display(fairness_df.round(3))
fairness_df.to_csv(CONFIG.output_dir / "fairness_audit.csv", index=False)
calibration_df.to_csv(CONFIG.output_dir / "calibration_summary.csv", index=False)
results_df.to_csv(CONFIG.output_dir / "ablation_results.csv", index=False)
results_df.to_csv(CONFIG.output_dir / "leaderboard_results.csv", index=False)

fairness_heatmap_df = fairness_df[fairness_df["group"] == "age_quartile"].pivot(index="model", columns="value", values="MAE")
if not fairness_heatmap_df.empty:
    fig, ax = plt.subplots(1, 1, figsize=(8, 4.5))
    sns.heatmap(fairness_heatmap_df, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax)
    ax.set_title("Age-Quartile MAE Audit")
    ax.set_xlabel("Age quartile")
    ax.set_ylabel("Model")
    plt.tight_layout()
    save_figure(fig, "fairness_age_quartile_heatmap")
    plt.show()

severity_heatmap_df = fairness_df[fairness_df["group"] == "baseline_md_quartile"].pivot(index="model", columns="value", values="MAE")
if not severity_heatmap_df.empty:
    fig, ax = plt.subplots(1, 1, figsize=(8, 4.5))
    sns.heatmap(severity_heatmap_df, annot=True, fmt=".3f", cmap="PuBuGn", ax=ax)
    ax.set_title("Baseline MD Quartile MAE Audit")
    ax.set_xlabel("Baseline MD quartile")
    ax.set_ylabel("Model")
    plt.tight_layout()
    save_figure(fig, "fairness_baseline_md_heatmap")
    plt.show()


def threshold_sensitivity_analysis(
    pred_dict: Dict[str, Dict[str, Dict[str, np.ndarray]]],
    model_names: Sequence[str],
    thresholds: Sequence[float] = (-0.5, -1.0, -1.5),
) -> pd.DataFrame:
    rows = []
    for model_name in model_names:
        bundle = pred_dict[model_name]["test"]
        risk_score = -bundle["slope_pred"]
        for threshold in thresholds:
            y_true = (bundle["y_slope"] <= threshold).astype(int)
            y_pred = (bundle["slope_pred"] <= threshold).astype(int)
            if len(np.unique(y_true)) < 2:
                continue
            rows.append(
                {
                    "Model": model_name,
                    "FastThreshold_dB_per_year": threshold,
                    "Prevalence": float(y_true.mean()),
                    "AUC_from_slope": safe_auc(y_true, risk_score),
                    "AP_from_slope": average_precision_score(y_true, risk_score),
                    "F1_from_slope_cutoff": f1_score(y_true, y_pred, zero_division=0),
                    "Precision_from_slope_cutoff": precision_score(y_true, y_pred, zero_division=0),
                    "Recall_from_slope_cutoff": recall_score(y_true, y_pred, zero_division=0),
                    "BalancedAcc_from_slope_cutoff": balanced_accuracy_score(y_true, y_pred),
                }
            )
    return pd.DataFrame(rows)


threshold_sensitivity_df = threshold_sensitivity_analysis(
    predictions,
    model_names=results_df["Model"].head(min(4, len(results_df))).tolist(),
)
display(threshold_sensitivity_df.round(3))
threshold_sensitivity_df.to_csv(CONFIG.output_dir / "threshold_sensitivity.csv", index=False)

if not threshold_sensitivity_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for model_name in threshold_sensitivity_df["Model"].unique():
        plot_df = threshold_sensitivity_df[threshold_sensitivity_df["Model"] == model_name].sort_values("FastThreshold_dB_per_year")
        axes[0].plot(
            plot_df["FastThreshold_dB_per_year"],
            plot_df["AUC_from_slope"],
            marker="o",
            lw=2.2,
            color=MODEL_COLORS.get(model_name),
            label=model_name,
        )
        axes[1].plot(
            plot_df["FastThreshold_dB_per_year"],
            plot_df["F1_from_slope_cutoff"],
            marker="s",
            lw=2.2,
            color=MODEL_COLORS.get(model_name),
            label=model_name,
        )
    axes[0].set_title("Threshold Sensitivity: AUC")
    axes[0].set_xlabel("Fast-progressor threshold (dB/year)")
    axes[0].set_ylabel("AUC from continuous slope prediction")
    axes[1].set_title("Threshold Sensitivity: F1")
    axes[1].set_xlabel("Fast-progressor threshold (dB/year)")
    axes[1].set_ylabel("F1 from slope cutoff")
    for ax in axes:
        ax.invert_xaxis()
        ax.legend(fontsize=9)
    plt.tight_layout()
    save_figure(fig, "threshold_sensitivity_panel")
    plt.show()


def fit_repeated_split_tabular_models(seed_offset: int) -> List[Dict[str, float]]:
    patient_df_repeat = build_patient_strata(all_prefix_samples)
    train_idx_r, temp_idx_r = balanced_group_shuffle_split(
        patient_df_repeat,
        test_size=0.20,
        random_state=SEED + 100 + seed_offset,
        n_candidates=96,
    )
    train_patients_r = set(patient_df_repeat.iloc[train_idx_r]["patient_id"])
    temp_patient_df_r = patient_df_repeat.iloc[temp_idx_r].reset_index(drop=True)
    val_idx_r, test_idx_r = balanced_group_shuffle_split(
        temp_patient_df_r,
        test_size=0.50,
        random_state=SEED + 200 + seed_offset,
        n_candidates=96,
    )
    val_patients_r = set(temp_patient_df_r.iloc[val_idx_r]["patient_id"])
    test_patients_r = set(temp_patient_df_r.iloc[test_idx_r]["patient_id"])

    train_samples_r = select_split_samples(all_prefix_samples, train_patients_r, split_name="train")
    val_samples_r = select_split_samples(all_prefix_samples, val_patients_r, split_name="val")
    test_samples_r = select_split_samples(all_prefix_samples, test_patients_r, split_name="test")

    normalizer_r = FeatureNormalizer()
    normalizer_r.fit(train_samples_r)
    X_train_r, y_train_reg_r, y_train_cls_r, _, _, _, _, w_train_r, _, _ = build_tabular_matrix(train_samples_r, normalizer_r)
    X_val_r, y_val_reg_r, y_val_cls_r, _, _, val_age_r, val_md_r, _, val_gender_r, _ = build_tabular_matrix(val_samples_r, normalizer_r)
    X_test_r, y_test_reg_r, y_test_cls_r, event_time_r, event_observed_r, test_age_r, test_md_r, _, test_gender_r, _ = build_tabular_matrix(test_samples_r, normalizer_r)

    rows = []
    model_predictions = {}

    if lgb is not None:
        reg = lgb.LGBMRegressor(
            objective="mae",
            n_estimators=450,
            learning_rate=0.035,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.8,
            random_state=SEED + seed_offset,
            verbose=-1,
        )
        clf = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=400,
            learning_rate=0.035,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=SEED + seed_offset,
            verbose=-1,
        )
        reg.fit(X_train_r, y_train_reg_r, sample_weight=w_train_r)
        clf.fit(X_train_r, y_train_cls_r, sample_weight=w_train_r)
        model_predictions["LightGBM_repeat"] = (reg.predict(X_val_r), clf.predict_proba(X_val_r)[:, 1], reg.predict(X_test_r), clf.predict_proba(X_test_r)[:, 1])

    if CatBoostRegressor is not None and CatBoostClassifier is not None:
        reg = CatBoostRegressor(
            loss_function="MAE",
            iterations=500,
            learning_rate=0.04,
            depth=6,
            random_seed=SEED + seed_offset,
            verbose=False,
        )
        clf = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=450,
            learning_rate=0.04,
            depth=6,
            auto_class_weights="Balanced",
            random_seed=SEED + seed_offset,
            verbose=False,
        )
        reg.fit(X_train_r, y_train_reg_r, sample_weight=w_train_r)
        clf.fit(X_train_r, y_train_cls_r, sample_weight=w_train_r)
        model_predictions["CatBoost_repeat"] = (reg.predict(X_val_r), clf.predict_proba(X_val_r)[:, 1], reg.predict(X_test_r), clf.predict_proba(X_test_r)[:, 1])

    reg = ExtraTreesRegressor(
        n_estimators=350,
        min_samples_leaf=2,
        random_state=SEED + seed_offset,
        n_jobs=MAX_CPU_THREADS,
    )
    clf = ExtraTreesClassifier(
        n_estimators=350,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=SEED + seed_offset,
        n_jobs=MAX_CPU_THREADS,
    )
    reg.fit(X_train_r, y_train_reg_r, sample_weight=w_train_r)
    clf.fit(X_train_r, y_train_cls_r, sample_weight=w_train_r)
    model_predictions["ExtraTrees_repeat"] = (reg.predict(X_val_r), clf.predict_proba(X_val_r)[:, 1], reg.predict(X_test_r), clf.predict_proba(X_test_r)[:, 1])

    for model_name, (val_slope, val_prob, test_slope, test_prob) in model_predictions.items():
        pred_bundle = {
            "y_slope": y_test_reg_r,
            "y_fast": y_test_cls_r,
            "event_time": event_time_r,
            "event_observed": event_observed_r,
            "baseline_age": test_age_r,
            "baseline_md": test_md_r,
            "slope_pred": test_slope,
            "fast_prob": test_prob,
            "gender": test_gender_r,
            "patient_id": np.asarray([sample["patient_id"] for sample in test_samples_r]),
            "eye_uid": np.asarray([sample["eye_uid"] for sample in test_samples_r]),
        }
        threshold = choose_threshold(y_val_cls_r, val_prob)
        rows.append({"split_seed": seed_offset, "Model": model_name, **summarize_predictions(pred_bundle, threshold)})
    return rows


if CONFIG.enable_repeated_tabular_validation:
    repeated_rows = []
    for split_seed in range(CONFIG.repeated_split_replicates):
        repeated_rows.extend(fit_repeated_split_tabular_models(split_seed))
    repeated_split_df = pd.DataFrame(repeated_rows)
    repeated_summary_df = (
        repeated_split_df
        .groupby("Model")
        .agg(
            MAE_mean=("MAE", "mean"),
            MAE_std=("MAE", "std"),
            AUC_mean=("AUC", "mean"),
            AUC_std=("AUC", "std"),
            AP_mean=("AP", "mean"),
            AP_std=("AP", "std"),
            F1_mean=("F1", "mean"),
            F1_std=("F1", "std"),
            Sensitivity_mean=("Sensitivity", "mean"),
            Sensitivity_std=("Sensitivity", "std"),
            Specificity_mean=("Specificity", "mean"),
            Specificity_std=("Specificity", "std"),
            C_index_mean=("C-index", "mean"),
            C_index_std=("C-index", "std"),
        )
        .reset_index()
    )
    display(repeated_summary_df.round(3))
    repeated_split_df.to_csv(CONFIG.output_dir / "repeated_split_tabular_results.csv", index=False)
    repeated_summary_df.to_csv(CONFIG.output_dir / "repeated_split_tabular_summary.csv", index=False)
else:
    repeated_split_df = pd.DataFrame()
    repeated_summary_df = pd.DataFrame()



## Cell 6 - Explainability and Visualization

Explainability is mandatory for a clinically serious model. We therefore visualize classical-feature SHAP values, spatial attention over the raw `8x9` VF, temporal attention over visits, and uncertainty-error relationships. The attention plots are intended as paper-ready mechanistic figures rather than post hoc decoration.



In [ ]:
def select_representative_case(pred_bundle: Dict[str, np.ndarray]) -> int:
    """Pick a clinically interesting test eye: fast progressor with large predicted risk."""
    fast_indices = np.where(pred_bundle["y_fast"] == 1)[0]
    if len(fast_indices) > 0:
        fast_probs = pred_bundle["fast_prob"][fast_indices]
        return int(fast_indices[np.argmax(fast_probs)])
    return int(np.argmax(np.abs(pred_bundle["y_slope"] - pred_bundle["slope_pred"])))


# SHAP for the strongest available tree-style baseline
if shap is not None:
    if boosting_name is not None and reg_baseline is not None:
        shap_model_name = boosting_name
        shap_model = reg_baseline
    elif catboost_reg is not None:
        shap_model_name = "CatBoost"
        shap_model = catboost_reg
    else:
        shap_model_name = "ExtraTrees"
        shap_model = extra_trees_reg

    shap_explainer = shap.TreeExplainer(shap_model)
    shap_values = shap_explainer.shap_values(X_test)

    top_feature_indices = np.argsort(np.abs(np.asarray(shap_values)).mean(axis=0))[::-1][:20]
    top_feature_names = [tab_feature_names[idx] for idx in top_feature_indices]

    plt.figure(figsize=(10, 7))
    shap.summary_plot(
        np.asarray(shap_values)[:, top_feature_indices],
        X_test[:, top_feature_indices],
        feature_names=top_feature_names,
        show=False,
    )
    plt.title(f"SHAP Summary ({shap_model_name} regression baseline)")
    plt.tight_layout()
    save_figure(plt.gcf(), f"shap_summary_{shap_model_name.lower().replace('-', '_')}")
    plt.show()
else:
    print("SHAP is not installed in this environment, so the SHAP summary plot is skipped.")


representative_idx = select_representative_case(predictions["ST-Transformer"]["test"])
single_item = test_dataset[representative_idx]
single_batch = collate_vf_sequences([single_item])
single_batch = move_batch_to_device(single_batch, DEVICE)

best_st_model.eval()
attention_output = best_st_model(single_batch, return_attention=True)
uncertainty_output = best_st_model.predict_with_uncertainty(single_batch, mc_passes=CONFIG.mc_dropout_passes)

td_spatial_pool_attn = attention_output["td_spatial_pool_attn"][0].detach().cpu().numpy()
hvf_spatial_pool_attn = attention_output["hvf_spatial_pool_attn"][0].detach().cpu().numpy()
fusion_gate = attention_output["fusion_gate"][0].detach().cpu().numpy()
temporal_attn = attention_output["temporal_cls_attn"][0].detach().cpu().numpy()
temporal_attn = temporal_attn.mean(axis=0)[: single_item["seq_len"]]

last_visit_td_attention = td_spatial_pool_attn[single_item["seq_len"] - 1].mean(axis=0).reshape(8, 9)
last_visit_td_attention = np.where(valid_mask, last_visit_td_attention, np.nan)
last_visit_hvf_attention = hvf_spatial_pool_attn[single_item["seq_len"] - 1].mean(axis=0).reshape(8, 9)
last_visit_hvf_attention = np.where(valid_mask, last_visit_hvf_attention, np.nan)
fusion_gate_strength = fusion_gate.mean(axis=-1)[: single_item["seq_len"]]

raw_last_td = test_samples[representative_idx]["visits"][-1]["td"]
raw_last_hvf = test_samples[representative_idx]["visits"][-1]["hvf"]
visit_ages = [visit["age"] for visit in test_samples[representative_idx]["visits"]]
visit_mds = [visit["md"] for visit in test_samples[representative_idx]["visits"]]

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
plot_vf_heatmap(axes[0], raw_last_td, "Representative TD (last observed prefix visit)", cmap="coolwarm", vmin=-35, vmax=5)
sns.heatmap(
    last_visit_td_attention,
    mask=np.isnan(last_visit_td_attention),
    cmap="magma",
    square=True,
    linewidths=0.25,
    linecolor="black",
    ax=axes[1],
)
axes[1].set_title("TD Spatial Attention")
axes[1].set_xlabel("Column")
axes[1].set_ylabel("Row")
sns.heatmap(
    last_visit_hvf_attention,
    mask=np.isnan(last_visit_hvf_attention),
    cmap="magma",
    square=True,
    linewidths=0.25,
    linecolor="black",
    ax=axes[2],
)
axes[2].set_title("HVF Spatial Attention")
axes[2].set_xlabel("Column")
axes[2].set_ylabel("Row")
axes[3].plot(range(1, len(temporal_attn) + 1), temporal_attn, marker="o", label="Temporal attention")
axes[3].plot(range(1, len(fusion_gate_strength) + 1), fusion_gate_strength, marker="s", label="Mean TD fusion gate")
axes[3].set_title("Temporal and Fusion Weights")
axes[3].set_xlabel("Visit index within prefix")
axes[3].set_ylabel("Weight")
axes[3].set_xticks(range(1, len(temporal_attn) + 1))
axes[3].legend()
plt.tight_layout()
save_figure(fig, "st_transformer_attention_panel")
plt.show()

print(
    {
        "true_future_slope": float(test_samples[representative_idx]["label_slope"]),
        "pred_future_slope": float(uncertainty_output["slope_mean"].cpu().numpy()[0]),
        "pred_fast_probability": float(uncertainty_output["fast_prob"].cpu().numpy()[0]),
        "predictive_std": float(uncertainty_output["slope_std"].cpu().numpy()[0]),
    }
)

st_test_uncertainty = predictions["ST-Transformer"]["test"]
abs_error = np.abs(st_test_uncertainty["y_slope"] - st_test_uncertainty["slope_pred"])
pred_std = st_test_uncertainty["slope_std"]
rho, pval = stats.spearmanr(pred_std, abs_error)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(pred_std, abs_error, alpha=0.7)
axes[0].set_title(f"Predictive Uncertainty vs. Absolute Error\nSpearman rho={rho:.3f}, p={pval:.2e}")
axes[0].set_xlabel("Predicted slope standard deviation")
axes[0].set_ylabel("Absolute slope error")

sorted_idx = np.argsort(pred_std)
running_error = pd.Series(abs_error[sorted_idx]).rolling(window=25, min_periods=5).mean()
axes[1].plot(pred_std[sorted_idx], running_error)
axes[1].set_title("Rolling Error vs. Uncertainty")
axes[1].set_xlabel("Predicted slope standard deviation")
axes[1].set_ylabel("Rolling mean absolute error")
plt.tight_layout()
save_figure(fig, "uncertainty_error_analysis")
plt.show()

uncertainty_df = pd.DataFrame(
    {
        "y_true_slope": st_test_uncertainty["y_slope"],
        "y_pred_slope": st_test_uncertainty["slope_pred"],
        "abs_error": abs_error,
        "pred_std": pred_std,
        "aleatoric_var": st_test_uncertainty["aleatoric_var"],
        "epistemic_var": st_test_uncertainty["epistemic_var"],
    }
)
uncertainty_df.to_csv(CONFIG.output_dir / "uncertainty_analysis.csv", index=False)



## Cell 7 - Results and Paper-Ready Outputs

This cell synthesizes abstract-ready text, exports final figures and tables, and writes a compact requirements snippet for exact reruns.



In [ ]:
st_metrics = results_df.set_index("Model").loc["ST-Transformer"]
best_metrics = results_df.set_index("Model").loc[best_overall_model]

abstract_summary = f"""
### Abstract-ready summary

- **Cohort/task**: patient-level hold-out evaluation on UWHVF with leakage-aware prefix-to-future prediction and post-hoc calibration from the validation split.
- **Best overall model**: **{best_overall_model}** achieved **MAE {best_metrics['MAE']:.3f} dB/year**, **AUC {best_metrics['AUC']:.3f}**, **AP {best_metrics['AP']:.3f}**, and **C-index {best_metrics['C-index']:.3f}**.
- **Flagship ST-Transformer**: the raw-grid dual-branch model achieved **MAE {st_metrics['MAE']:.3f} dB/year**, **RMSE {st_metrics['RMSE']:.3f} dB/year**, **AUC {st_metrics['AUC']:.3f}**, and **C-index {st_metrics['C-index']:.3f}**.
- **Clinical translation**: the notebook reports calibrated discrimination, accuracy, sensitivity/recall, specificity, precision, F1, event-based ranking, subgroup robustness by age, sex, and baseline MD severity, uncertainty analysis, threshold sensitivity, repeated split robustness, and decision-curve utility from a single reproducible pipeline.
- **Methodological novelty**: this is a reproducible spatio-temporal glaucoma progression benchmark that combines raw `8x9` VF grids, longitudinal temporal attention, derived glaucoma biomarkers, and post-hoc uncertainty-aware reporting in one Kaggle-ready workflow.
"""
display(Markdown(textwrap.dedent(abstract_summary)))


fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

plot_df = results_df.head(min(6, len(results_df))).copy().iloc[::-1]
axes[0].barh(plot_df["Model"], plot_df["MAE"], color=[MODEL_COLORS.get(name, "#4C78A8") for name in plot_df["Model"]])
axes[0].set_title("MAE (lower is better)")
axes[0].set_xlabel("MAE (dB/year)")

axes[1].barh(plot_df["Model"], plot_df["AUC"], color=[MODEL_COLORS.get(name, "#4C78A8") for name in plot_df["Model"]])
axes[1].set_title("AUC (higher is better)")
axes[1].set_xlabel("AUC")

axes[2].barh(
    plot_df["Model"],
    plot_df["CompositeRank"],
    color=[MODEL_COLORS.get(name, "#4C78A8") for name in plot_df["Model"]],
)
axes[2].set_title("Composite Rank (lower is better)")
axes[2].set_xlabel("Composite rank")

plt.tight_layout()
save_figure(fig, "leaderboard_barplot")
plt.show()


fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Predicted vs true slope for the best overall model
axes[0].scatter(best_test["y_slope"], best_test["slope_pred"], alpha=0.65, color=MODEL_COLORS.get(best_overall_model, "#0B3C5D"))
lims = [
    min(best_test["y_slope"].min(), best_test["slope_pred"].min()),
    max(best_test["y_slope"].max(), best_test["slope_pred"].max()),
]
axes[0].plot(lims, lims, linestyle="--", color="black")
axes[0].set_title(f"Predicted vs. True Future MD Slope ({best_overall_model})")
axes[0].set_xlabel("True future slope (dB/year)")
axes[0].set_ylabel("Predicted future slope (dB/year)")

# Representative progression curve
representative_sample = test_samples[representative_idx]
prefix_ages = np.asarray([visit["age"] for visit in representative_sample["visits"]], dtype=np.float32)
prefix_mds = np.asarray([visit["md"] for visit in representative_sample["visits"]], dtype=np.float32)
future_true_slope = representative_sample["label_slope"]
future_start_age = prefix_ages[-1]
future_grid = np.linspace(future_start_age, future_start_age + 5.0, 50)
predicted_line = prefix_mds[-1] + (future_grid - future_start_age) * best_test["slope_pred"][representative_idx]
true_line = prefix_mds[-1] + (future_grid - future_start_age) * future_true_slope

axes[1].plot(prefix_ages, prefix_mds, marker="o", color="#444444", label="Observed prefix MD")
axes[1].plot(future_grid, predicted_line, color=MODEL_COLORS.get(best_overall_model, "#0B3C5D"), lw=2.5, label=f"{best_overall_model} prediction")
axes[1].plot(future_grid, true_line, linestyle="--", color="#B22222", lw=2.0, label="True future slope")
axes[1].set_title("Representative Eye: Forecasted Progression")
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("MD (dB)")
axes[1].legend()

plt.tight_layout()
save_figure(fig, "paper_ready_overview")
plt.show()


results_with_p = results_df.merge(pvalue_df, on="Model", how="left")
latex_table_with_p = results_with_p.round(4).to_latex(
    index=False,
    escape=False,
    caption=f"Final ablation table with paired significance tests versus the best overall model ({reference_model}).",
)
with open(CONFIG.output_dir / "ablation_results_with_pvalues.tex", "w", encoding="utf-8") as f:
    f.write(latex_table_with_p)

print(latex_table_with_p)

requirements_snippet = """
torch>=2.2.0
numpy>=1.26.0
pandas>=2.2.0
scikit-learn>=1.4.0
scipy>=1.12.0
matplotlib>=3.8.0
seaborn>=0.13.0
optuna>=3.6.0
shap>=0.45.0
lightgbm>=4.3.0
xgboost>=2.0.0
catboost>=1.2.0
lifelines>=0.29.0
"""

prediction_frames = []
for model_name, bundle in predictions.items():
    for split_name in ["val", "test"]:
        split_bundle = bundle[split_name]
        prediction_frames.append(
            pd.DataFrame(
                {
                    "model": model_name,
                    "split": split_name,
                    "patient_id": split_bundle["patient_id"],
                    "eye_uid": split_bundle["eye_uid"],
                    "baseline_age": split_bundle["baseline_age"],
                    "baseline_md": split_bundle["baseline_md"],
                    "true_slope": split_bundle["y_slope"],
                    "pred_slope": split_bundle["slope_pred"],
                    "true_fast": split_bundle["y_fast"],
                    "pred_fast_prob": split_bundle["fast_prob"],
                    "event_time": split_bundle["event_time"],
                    "event_observed": split_bundle["event_observed"],
                }
            )
        )
prediction_table = pd.concat(prediction_frames, ignore_index=True)
prediction_table.to_csv(CONFIG.output_dir / "prediction_table_long.csv", index=False)

split_manifest = {
    "resolved_data_path": str(CONFIG.data_path),
    "train_patients": sorted({sample["patient_id"] for sample in train_samples}),
    "val_patients": sorted({sample["patient_id"] for sample in val_samples}),
    "test_patients": sorted({sample["patient_id"] for sample in test_samples}),
    "train_eyes": sorted({sample["eye_uid"] for sample in train_samples}),
    "val_eyes": sorted({sample["eye_uid"] for sample in val_samples}),
    "test_eyes": sorted({sample["eye_uid"] for sample in test_samples}),
    "n_train_samples": len(train_samples),
    "n_val_samples": len(val_samples),
    "n_test_samples": len(test_samples),
    "n_train_eyes": len({sample["eye_uid"] for sample in train_samples}),
    "n_val_eyes": len({sample["eye_uid"] for sample in val_samples}),
    "n_test_eyes": len({sample["eye_uid"] for sample in test_samples}),
    "validation_test_prefix_policy": "earliest admissible prefix per eye",
    "train_prefix_policy": "all eligible prefixes" if CONFIG.train_all_prefixes else "one prefix per eye",
}
with open(CONFIG.output_dir / "split_manifest.json", "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, indent=2)

final_manifest = copy.deepcopy(RUN_MANIFEST)
final_manifest["best_overall_model"] = best_overall_model
final_manifest["reference_model_for_tests"] = reference_model
final_manifest["topline_results"] = {
    row["Model"]: {
        "MAE": float(row["MAE"]),
        "AUC": float(row["AUC"]),
        "AP": float(row["AP"]),
        "C-index": float(row["C-index"]),
        "CompositeRank": float(row["CompositeRank"]),
    }
    for _, row in results_df.iterrows()
}
if "repeated_summary_df" in globals() and not repeated_summary_df.empty:
    final_manifest["repeated_split_summary"] = repeated_summary_df.to_dict(orient="records")
with open(CONFIG.output_dir / "final_run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(final_manifest, f, indent=2)

display(
    Markdown(
        textwrap.dedent(
            f"""
            ### Reproducibility note

            - Random seeds were fixed across Python, NumPy, and PyTorch.
            - The notebook auto-resolved the dataset path and output directory for both local runs and Kaggle.
            - Splitting was patient-level to prevent leakage across fellow eyes and repeated prefixes.
            - Post-hoc calibration was fit on validation predictions only, then applied once to the held-out test set.
            - Model weights, Optuna trials, calibration summaries, threshold-sensitivity analyses, repeated split robustness tables, long-form predictions, LaTeX tables, and publication figures were exported to `{CONFIG.output_dir}`.
            - The resolved compute device for this run was `{DEVICE.type}`.
            - When optional packages are unavailable, the notebook skips only the corresponding non-core extras or uses a built-in fallback instead of crashing.

            ### `requirements.txt` snippet

                {requirements_snippet.strip()}
            """
        )
    )
)

with open(CONFIG.output_dir / "abstract_ready_summary.md", "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(abstract_summary))
with open(CONFIG.output_dir / "requirements_snippet.txt", "w", encoding="utf-8") as f:
    f.write(requirements_snippet.strip() + "\n")



## Next Steps for Publication

To strengthen a TVST / Ophthalmology Science / TBME submission, prioritize the following next analyses and figures:

- Add a sensitivity analysis for different fast-progressor thresholds, such as `-0.5`, `-1.0`, and `-1.5 dB/year`.
- Include a robustness table for minimum prefix length, minimum future span, and censoring definitions.
- Compare TD-only, HVF-only, and TD+HVF fusion to isolate the value of multimodal raw-grid encoding.
- Report subgroup calibration and net benefit by baseline severity strata, age quartiles, and sex where sample size permits.
- Add a figure showing spatial attention maps for slow versus fast progressors, aligned to clinically recognizable arcuate loss patterns.
- Include bootstrap confidence intervals for MAE, AUC, AP, and C-index.
- Consider a mixed-effects or generalized estimating equation sensitivity analysis to account explicitly for bilateral correlation.
- Add a short clinical impact paragraph emphasizing uncertainty-aware triage, earlier detection of aggressive progression, and reproducibility across commodity CPU settings and Kaggle accelerators.
